# SkinSight GPU training
Enable **GPU** and **Internet** in Kaggle settings, then Run All. This notebook contains the corrected app and training code; no Git push is needed. It downloads several GB of public datasets. Full-data/GPU execution was not run in the development workspace. Dataset source attribution and terms apply. Results are research metrics, not clinical accuracy. Do not repeatedly tune using the held-out test. Official ISIC metadata provides lesion groups where present. Lesion groups do not establish patient separation. Exact-file hashes also prevent identical files crossing splits. The previously reported test is excluded from this fine-tuning experiment; new external data is required for another independent accuracy claim.

In [ ]:
import os, sys, subprocess
from pathlib import Path
os.environ['NO_ALBUMENTATIONS_UPDATE'] = '1'
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU accelerator before running this notebook.')
print('GPU:', torch.cuda.get_device_name(0))
# Preserve Kaggle's matching CUDA torch/torchvision packages.
subprocess.run([sys.executable, '-m', 'pip', 'install', 'kagglehub==0.3.12',
    'fastapi==0.115.12', 'uvicorn==0.34.2', 'python-multipart==0.0.20',
    'jinja2==3.1.6', 'albumentations==2.0.7', 'pandas==2.2.3',
    'scikit-learn==1.6.1', 'opencv-python-headless==4.11.0.86'], check=True)
import torchvision
print('torch:',torch.__version__, 'torchvision:',torchvision.__version__)


In [ ]:
PROJECT = Path('/kaggle/working/SkinSight_AI')
PROJECT.mkdir(parents=True, exist_ok=True)
FILES = {'DEVELOPMENT_STATUS.md': '# Current development status\n\nFull-model fine-tuning and PAD adaptation are complete. The selected model is connected to the app. See [fine-tuning results](reports/finetune_v1/RESULTS.md). PAD validation balanced accuracy is 56.9%, compared with 50.4% for the warm-start baseline on the same validation cohort. No new independent-test accuracy is established.\n\nThirty-two regression tests passed. Browser checks verified the new public pages, dashboard, saved results, light/dark themes and 390-pixel mobile layouts; actual skin-photo prediction, PDF download and avatar rejection passed. Account-owned PDF reports and direct-to-registered-email automatic Gmail report delivery are implemented; actual Gmail sending needs a configured sender App Password and remains untested live. Registration/login/logout, expiring server-side sessions, CSRF checks, conservative upload-quality rejection and stage education are implemented. These quality checks are not a validated semantic skin detector. External-cohort overlap auditing, evaluation and temperature calibration commands are implemented. Actual external clinical evaluation and calibration still require a new eligible labeled dataset. The old test partition is excluded from the new training runs.\n\nThe checkpoint is models/skinsight_mobilenetv3_small.pt, SHA256 a4619d1b9f73cdf2fa415e285642878472356cdd3a135dd8e6cd316d629db0d1. Code and aggregate reports are stored in Git; weights are included in the downloadable trained app. Live processes must restart in future tasks. Fresh-task restoration has not been independently verified.\n', 'README.md': '# SkinSight AI\n\nEducational skin-lesion classification using **MobileNetV3-Small**. Six classes:\nACK, BCC, MEL, NEV, SCC, SEK. ISIC AK maps to ACK, NV to NEV, BKL to SEK.\nBKL is broader than PAD\'s seborrheic keratosis, so this mapping is approximate.\n\n## Easiest GPU training workflow\n\nImport `notebooks/train_kaggle.ipynb` into Kaggle. Enable an available GPU and\nInternet, then Run All. The notebook bundles this code, downloads the public\nPAD and resized ISIC datasets plus ground truth, validates matching, trains,\nand exports `skinsight_trained_app.zip`. GPU/account access and full-data downloads\nhave not been verified here. Follow dataset licenses and attribution requirements.\n\n## Windows local workflow\n\nRun these commands from `SkinSight_AI` (Python 3.12 recommended):\n\n```powershell\npy -m venv .venv\n.\\.venv\\Scripts\\python -m pip install torch==2.7.0 torchvision==0.22.0 --index-url https://download.pytorch.org/whl/cpu\n.\\.venv\\Scripts\\python -m pip install -r requirements.txt\n.\\.venv\\Scripts\\python ml\\prepare_combined.py --pad-meta "D:\\SkinCancerData\\PAD-UFES-20\\extracted\\metadata.csv" --pad-root "D:\\SkinCancerData\\PAD-UFES-20\\extracted" --isic-gt "D:\\SkinCancerData\\ISIC-2019\\ISIC_2019_Training_GroundTruth.csv" --isic-root "D:\\SkinCancerData\\ISIC-2019\\extracted\\train-image" --out data\\combined_manifest.csv\n.\\.venv\\Scripts\\python ml\\train.py --manifest data\\combined_manifest.csv --epochs 10 --batch-size 16 --size 160 --fine-tune --device auto --output-dir models\n.\\.venv\\Scripts\\python run.py\n```\n\nPAD image folders are scanned recursively. Missing images cause an error instead\nof a partial manifest. CPU training on a low-memory laptop is likely slow.\n`--device auto` selects CUDA if available; `--device cuda` requires a CUDA GPU.\nPretrained weights download from PyTorch. Without `--fine-tune`, the backbone and\nits BatchNorm statistics stay frozen. `--from-scratch` requires `--fine-tune`.\n\nOutputs: `models/skinsight_mobilenetv3_small.pt`, `metrics.json`, `train_split.csv`,\n`validation_split.csv`. Reports include per-source validation metrics.\n\n## Connect trained weights\n\nCopy the trained checkpoint to `SkinSight_AI/models/skinsight_mobilenetv3_small.pt`\nand restart the app. Alternatively set `SKINSIGHT_MODEL_PATH` to an absolute path.\nOnly use trusted checkpoints; loading uses `weights_only=True`.\nWithout a checkpoint `/api/health` reports `model_ready=false` and predictions\nreturn HTTP 503. Synthetic test weights must never be deployed.\n\n## Evaluation limits\n\nPAD groups by real patient IDs. ISIC ground truth has no patient IDs; official\nmetadata supplies lesion groups where present. Lesion or image grouping **does not\nestablish patient-level separation**. The resized mirror\'s `dummy_*` IDs are not\ngenuine patient IDs. Validation selects the checkpoint; a separate internal test\npartition evaluates it afterward. Genuine ISIC patient metadata, probability\ncalibration, and external smartphone testing remain needed. Source/class balancing cannot\neliminate clinical-versus-dermoscopic domain shift. The 224px mirror is a practical\nbaseline; original ISIC images contain more detail.\n\nThis app is an educational research tool, not a diagnosis or clinical staging\nsystem. A real-data frozen-feature baseline has now been trained; see reports/baseline_v1/RESULTS.md. It has not been clinically validated.\n\n## Tests\n\n```powershell\n.\\.venv\\Scripts\\python -m unittest discover -s tests -v\n```\n\nRegression tests use artificial images to check software behavior only.\n\n## Independent holdout and grouping\n\nTraining now creates a fixed, source/class-stratified **train/validation/test**\npartition (approximately 60/20/20). Linked patient IDs, lesion IDs, image IDs,\nand exact-file SHA-256 duplicates stay in a single partition. Each partition must\ncontain every source/class represented in the manifest; otherwise training fails\nwith a diagnostic rather than silently reporting incomplete results.\n\nSupply `--isic-meta path/to/ISIC_2019_Training_Metadata.csv` to preparation to\nuse authoritative ISIC lesion IDs and genuine patient IDs where available.\nThe Kaggle notebook downloads the official metadata automatically. Dummy patient\nIDs are ignored. `split_summary.json` records grouping coverage; missing IDs remain\nexplicit image-level groups. File hashes detect exact duplicates, not re-encoded\nor cropped copies. Lesion separation still does not prove patient separation.\n\nThe test partition is never used to select checkpoints. After training, the best\nvalidation checkpoint is loaded and test metrics are computed once. Outputs now\nalso include `test_split.csv`, `split_summary.json`, and `test_metrics.json`.\nOverall and per-source test reports contain confusion matrices, per-class\nsensitivity/specificity, one-vs-rest ROC AUC (null when undefined), and calibration\nerror. PAD-only test metrics are the relevant internal smartphone-photo measure.\nExternal smartphone performance remains unknown until separately evaluated.\n\nUse a fresh output directory for another run. Reusing test feedback to tune a\nmodel contaminates the test set even when filenames are different. Existing\ncheckpoints/test results will not be overwritten automatically.\n\n## Fast CPU baseline\n\nUse `--cache-features` without `--fine-tune` or `--from-scratch` to extract frozen\npretrained MobileNet features once and train only its classifier. For example:\n\n```powershell\n.\\.venv\\Scripts\\python ml\\train.py --manifest data\\combined_manifest.csv --epochs 5 --batch-size 64 --size 160 --cache-features --device cpu --output-dir models_baseline\n```\n\nThis mode uses deterministic resize/normalization without training augmentation.\nIt is a quick baseline, not full-model fine-tuning. Cached features are kept in RAM;\nvalidation/test partitioning and checkpoint selection rules remain unchanged.\nThe resulting checkpoint still loads through the same full MobileNet predictor.\n\n## Fine-tuning without reusing the reported test\n\nWarm-start from the baseline and select using PAD smartphone validation:\n\n```powershell\n.\\.venv\\Scripts\\python ml\\train.py --manifest data\\combined_manifest.csv --init-checkpoint models\\skinsight_mobilenetv3_small.pt --fine-tune --epochs 5 --lr 0.00003 --batch-size 32 --size 160 --selection-source PAD-UFES-20 --skip-test --output-dir models_finetune\n```\n\n`--skip-test` never loads test images or computes test metrics. It still preserves\nsplit identity records for auditing. The warm-start checkpoint is evaluated as epoch\nzero and retained if no fine-tuned epoch improves the chosen validation score.\nSource-specific selection scores are labeled; they must not be confused with the\ncombined validation score or old held-out scores. Full-model training is slower\nthan cached classifier-head training, particularly on a CPU.\n\n## New external evaluation and calibration\n\nProvide a new, independently collected labeled manifest with columns\n`image_path,label,source,original_id,image_sha256,patient_id,lesion_id`. Labels must\ncome from authoritative ground truth and map to the supported six classes; do not\ninvent labels. SHA-256 must match each actual image. Missing patient identifiers\nlimit independence claims; genuine patient IDs are required for calibration.\n\n```powershell\n.\\.venv\\Scripts\\python ml\\evaluate_external.py --manifest external_test.csv --checkpoint models_finetune\\skinsight_mobilenetv3_small.pt --reference-dir models_baseline --output external_report.json\n.\\.venv\\Scripts\\python ml\\calibrate.py --manifest new_calibration.csv --checkpoint models_finetune\\skinsight_mobilenetv3_small.pt --reference-dir models_baseline --output-checkpoint models_finetune\\calibrated.pt\n```\n\nBoth tools reject overlap with the recorded training, validation, and reported test\nidentities and exact file hashes. Aliased identities and re-encoded images still\nneed a cohort/provenance review. Calibration uses a **new cohort**, not the old\nreported test. Temperature scaling changes model scores but not class ordering or\naccuracy. NLL improvement on calibration data is not proof of performance on a new\ntest. Clinical confidence remains unvalidated.\n\nThe calibrated checkpoint has a companion `.calibration.csv` identity file. Keep it\nprivate and alongside the checkpoint when evaluating a new external test: the\nexternal evaluation command requires it and also rejects calibration/test overlap.\nDo not upload identity CSVs or patient images to GitHub. Only aggregate reports may\nbe published. The app accepts positive finite temperature metadata automatically.\n\nA candidate independently collected clinical cohort is Stanford\'s DDI (656 images,\n570 patients, biopsy-grounded diagnoses). Its official documentation requires each\nuser to register and agree to its research-use terms. Access instructions are at\nhttps://ddi-dataset.github.io/#access and the official portal is linked there.\nDDI was **not downloaded or evaluated** in this workspace. Its supported categories,\npatient identifiers, and permitted use must be checked before evaluation. If patient\nidentifiers are unavailable, do not randomly split its images into supposedly\npatient-independent calibration and test sets.\n\n## Upload quality checks\n\nThe API rejects tiny images, blank/extremely exposed images, grayscale inputs and obvious flat-color graphics before classification (HTTP 422, no prediction). Changing images or receiving an error clears the old result. These conservative heuristics are not a trained skin/lesion detector: unrelated natural photos may still pass, and legitimate images may be rejected. Do not treat acceptance as proof that a lesion is present. A validated semantic detector needs representative skin and non-skin training and evaluation data.\n\n## Accounts and login\n\nOn Windows, double-click `start_windows.bat` from the extracted app folder, or use the existing PowerShell setup commands. Open the local app, choose **Create an account**, register a username and a password of at least 12 characters, then log in. Logout revokes the session. Accounts persist in `data/accounts.sqlite3` on the machine hosting this app; separate computers have separate accounts. Keep this database private and outside shared ZIPs/Git. Uploaded photos are processed in memory; screening report summaries are saved privately for downloads.\n\nPasswords use salted PBKDF2-SHA256 hashes; sessions are random server-side records with an eight-hour expiry. Login attempts are rate limited, and POST operations require CSRF tokens. This is a local demo with direct report email delivery and no password recovery. For HTTPS hosting set `SKINSIGHT_SECURE_COOKIES=1`; internet deployment requires its own operational review. The health endpoint is public, but the screening page and prediction API require login.\n\nCancer stage information is educational. The app never assigns a clinical stage from an image. For melanoma it explains stages 0–IV; for BCC/SCC it explains the different assessment requirements. Stage determination needs confirmed diagnosis and clinical/pathology findings.\n\n## Download reports and email through Gmail\n\nEvery successful analysis creates an account-owned PDF screening report with model scores, doctor guidance and stage limitations. Choose **Download PDF report** below the result. Registered email addresses receive reports directly, without verification codes. Report summaries remain private to the owning account; uploaded images are not attached. Rejected images create no report or email.\n\n1. Enable Google 2-Step Verification for the sender Gmail account and create a Google App Password: https://support.google.com/accounts/answer/185833.\n2. Stop any running app with Ctrl+C, then double-click **start_with_gmail.bat**. Enter the sender Gmail address and its App Password at the masked local prompt. This is a one-time setup: the launcher saves a Windows DPAPI-encrypted credential at `data/gmail-credentials.clixml`, readable only by the same Windows user on the same computer. Future starts load it automatically. To change credentials, stop the app, delete that file and restart. Never share the data folder.\n3. Register with your report email and log in using username or email. Existing users can add or change their recipient address under **Email settings**. No verification step is required.\n4. Analyze a skin photo. The app sends the PDF and doctor guidance directly to the account email, then displays the sending status. Downloads still work if sending fails.\n\nGmail uses verified TLS at smtp.gmail.com:465. Environment variables are SKINSIGHT_SMTP_HOST, SKINSIGHT_SMTP_PORT, SKINSIGHT_SMTP_USER, SKINSIGHT_SMTP_PASSWORD (App Password), and SKINSIGHT_MAIL_FROM. Sending and receiving accounts may be different. Gmail acceptance does not guarantee inbox delivery. Sending is best effort, not a durable queue. Live Gmail delivery remains untested here without sender credentials. Windows launcher execution is not verified in this Linux environment.\n\nReports contain educational screening information, not a diagnosis or clinical cancer stage. Keep databases and sender credentials private and outside shared ZIPs or Git.\n\n## Site pages and saved-report dashboard\n\nThe existing root URL `/` keeps its authenticated analysis workflow; `/analyze` is an alias. Added pages are `/home`, `/about`, `/faq` and `/stages` (public), `/dashboard` (your account\'s saved report summaries) and `/reports/{report_id}` (owner-only saved result). Navigation links connect the pages. The interface includes responsive layouts, light/dark theme selection, drag-and-drop upload and a loading state; prediction, login, PDF and direct-email behavior are retained.\n\nDashboard counts and charts use actual account reports, with an empty state for new accounts. Cancer-class predictions are labeled as model outputs, not diagnoses or clinical risk totals. Images remain unpersisted, so saved reports have no image thumbnails. Stage guide content is educational; no stage is predicted from an image.\n\nWhen updating an existing Windows installation, stop the server, back up the folder, and replace app files from the new ZIP. Preserve the existing `.venv` and private `data` directory to keep accounts/reports and the encrypted Gmail sender settings. Do not share `data` or credentials with teammates. Start with the same existing `start_with_gmail.bat` workflow.\n', 'requirements.txt': 'fastapi==0.115.12\nuvicorn[standard]==0.34.2\npython-multipart==0.0.20\njinja2==3.1.6\npillow==11.2.1\nnumpy==2.2.5\ntorch==2.7.0\ntorchvision==0.22.0\ntimm==1.0.15\npandas==2.2.3\nscikit-learn==1.6.1\nalbumentations==2.0.7\nopencv-python-headless==4.11.0.86\n', 'run.py': 'import argparse\nimport uvicorn\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--host", default="127.0.0.1")\n    parser.add_argument("--port", type=int, default=8000)\n    parser.add_argument("--reload", action="store_true")\n    args = parser.parse_args()\n    uvicorn.run("app.main:app", host=args.host, port=args.port, reload=args.reload)\n', 'start_with_gmail.ps1': "param([switch]$Reset)\n$ErrorActionPreference = 'Stop'\nSet-Location $PSScriptRoot\n# Export-Clixml protects PSCredential passwords using Windows DPAPI.\n# Only this Windows user on this computer can decrypt the saved password.\n$gmailConfigPath = Join-Path $PSScriptRoot 'data\\gmail-credentials.clixml'\nif ($Reset -and (Test-Path $gmailConfigPath)) { Remove-Item $gmailConfigPath }\nif (Test-Path $gmailConfigPath) {\n    try {\n        $gmailCredential = Import-Clixml $gmailConfigPath\n        if ($gmailCredential -isnot [System.Management.Automation.PSCredential]) { throw 'Invalid credential file' }\n    } catch {\n        throw 'Cannot read saved Gmail settings. Delete data\\gmail-credentials.clixml and run again to configure this Windows account.'\n    }\n} else {\n    $gmailSender = (Read-Host 'One-time setup: sending Gmail address').Trim()\n    if ($gmailSender -notmatch '^[^\\s@]+@gmail\\.com$') { throw 'Enter the sending Gmail address.' }\n    $gmailSecret = Read-Host 'Google App Password (not your normal Gmail password)' -AsSecureString\n    $gmailCredential = New-Object System.Management.Automation.PSCredential($gmailSender, $gmailSecret)\n    New-Item -ItemType Directory -Path (Split-Path $gmailConfigPath) -Force | Out-Null\n    $gmailCredential | Export-Clixml -Path $gmailConfigPath\n    Write-Host 'Gmail settings saved encrypted for this Windows account. Future starts will not prompt.'\n}\n$gmailPointer = [Runtime.InteropServices.Marshal]::SecureStringToBSTR($gmailCredential.Password)\ntry {\n    $env:SKINSIGHT_SMTP_HOST = 'smtp.gmail.com'\n    $env:SKINSIGHT_SMTP_PORT = '465'\n    $env:SKINSIGHT_SMTP_USER = $gmailCredential.UserName\n    $env:SKINSIGHT_MAIL_FROM = $gmailCredential.UserName\n    $env:SKINSIGHT_SMTP_PASSWORD = [Runtime.InteropServices.Marshal]::PtrToStringBSTR($gmailPointer).Replace(' ', '')\n    Write-Host 'Open http://127.0.0.1:8000. Reports are emailed directly to the registered address after analysis.'\n    & '.\\.venv\\Scripts\\python.exe' run.py\n} finally {\n    [Runtime.InteropServices.Marshal]::ZeroFreeBSTR($gmailPointer)\n    Remove-Item Env:SKINSIGHT_SMTP_PASSWORD -ErrorAction SilentlyContinue\n    $gmailCredential.Password.Dispose()\n}\n", 'start_with_gmail.bat': '@echo off\nsetlocal\ncd /d "%~dp0"\nif not exist ".venv\\Scripts\\python.exe" (\n  py -3 -m venv .venv\n  if errorlevel 1 goto failed\n)\n".venv\\Scripts\\python.exe" -m pip install -r requirements.txt\nif errorlevel 1 goto failed\npowershell -NoProfile -ExecutionPolicy Bypass -File "%~dp0start_with_gmail.ps1"\npause\nexit /b\n:failed\necho Setup failed. Copy the error above for help. Python 3.12 is recommended.\npause\nexit /b 1\n', 'start_windows.bat': '@echo off\nsetlocal\ncd /d "%~dp0"\nif not exist ".venv\\Scripts\\python.exe" (\n  py -3 -m venv .venv\n  if errorlevel 1 goto failed\n)\n".venv\\Scripts\\python.exe" -m pip install -r requirements.txt\nif errorlevel 1 goto failed\necho.\necho Open http://127.0.0.1:8000 in your browser.\necho Create an account, then log in. Keep this window open.\n".venv\\Scripts\\python.exe" run.py\npause\nexit /b\n:failed\necho.\necho Setup failed. Copy the error above and share it for help.\necho This app requires a supported Python version; Python 3.12 is recommended.\npause\nexit /b 1\n', 'app/guidance.py': 'GUIDANCE = {\n    "ACK": {\n        "name": "Actinic Keratosis",\n        "risk": "Needs dermatologist review",\n        "message": "Often related to sun damage and can be precancerous. Arrange a dermatologist review.",\n        "urgency": "Book a routine-to-soon dermatology appointment, especially if changing, tender, crusting, or bleeding."\n    },\n    "BCC": {\n        "name": "Basal Cell Carcinoma",\n        "risk": "High concern",\n        "message": "This class represents a skin cancer category in the training dataset.",\n        "urgency": "Arrange prompt in-person dermatology assessment and biopsy consideration."\n    },\n    "MEL": {\n        "name": "Melanoma",\n        "risk": "High concern",\n        "message": "This class represents melanoma, which requires professional assessment.",\n        "urgency": "Arrange prompt dermatologist evaluation. Do not rely on this image result to confirm or rule out melanoma."\n    },\n    "NEV": {\n        "name": "Melanocytic Nevus",\n        "risk": "Usually lower concern",\n        "message": "Many nevi are benign, but changing or unusual lesions still need examination.",\n        "urgency": "Seek review if the lesion is new, changing, asymmetric, bleeding, itching, painful, or looks different from your other moles."\n    },\n    "SCC": {\n        "name": "Squamous Cell Carcinoma",\n        "risk": "High concern",\n        "message": "This class represents a skin cancer category in the training dataset.",\n        "urgency": "Arrange prompt in-person dermatology assessment and biopsy consideration."\n    },\n    "SEK": {\n        "name": "Benign / Seborrheic Keratosis",\n        "risk": "Usually lower concern",\n        "message": "This is commonly benign, but visual similarity with other lesions can occur.",\n        "urgency": "Seek clinical review if rapidly changing, bleeding, painful, or uncertain."\n    },\n}\n\ndef stage_info(label: str):\n    if label in {"BCC", "MEL", "SCC"}:\n        return {\n            "status": "Not determinable from a photo",\n            "education": ([\n                "Melanoma stage overview — educational only; no stage is assigned to this image.",\n                "Stage 0: melanoma in situ, limited to the outer skin layer.",\n                "Stages I–II: localized melanoma; pathology including thickness and ulceration distinguishes stages.",\n                "Stage III: regional lymph nodes or nearby skin/lymphatic spread.",\n                "Stage IV: spread to distant sites or organs.",\n            ] if label == "MEL" else [\n                "A clinician must confirm the diagnosis and determine whether formal staging applies.",\n                "BCC is usually assessed by local extent and recurrence risk; routine numerical staging is uncommon.",\n                "SCC staging depends on the anatomic site, tumor features and evidence of lymph-node or distant spread.",\n            ]),\n            "explanation": (\n                "Clinical cancer Stage I–IV cannot be assigned safely from this image alone. "\n                "Staging requires pathology/biopsy and may require tumor depth, ulceration, "\n                "lymph-node findings, imaging, or evidence of spread."\n            ),\n        }\n    return {\n        "status": "Not applicable from image classification",\n        "explanation": (\n            "The image model provides lesion-class screening, not clinical cancer staging."\n        ),\n    }\n', 'app/inference.py': 'from pathlib import Path\nimport io\nimport os\nimport math\nimport numpy as np\nfrom PIL import Image\nimport torch\nfrom torchvision.models import mobilenet_v3_small\nfrom torch import nn\nimport albumentations as A\nfrom albumentations.pytorch import ToTensorV2\n\nPROJECT_ROOT = Path(__file__).resolve().parents[1]\nCHECKPOINT = Path(os.environ.get("SKINSIGHT_MODEL_PATH", str(PROJECT_ROOT / "models/skinsight_mobilenetv3_small.pt")))\n\nclass Predictor:\n    def __init__(self):\n        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n        self.model = None\n        self.class_names = None\n        self.image_size = 160\n        self.temperature = 1.0\n        self.meta = {}\n        self._load()\n\n    def _load(self):\n        if not CHECKPOINT.exists():\n            return\n        ckpt = torch.load(CHECKPOINT, map_location=self.device, weights_only=True)\n        self.temperature = float(ckpt.get("temperature", 1.0))\n        if not math.isfinite(self.temperature) or self.temperature <= 0:\n            raise ValueError("Checkpoint temperature must be finite and positive")\n        self.class_names = ckpt["class_names"]\n        self.image_size = int(ckpt.get("image_size", 160))\n        self.meta = {\n            "architecture": ckpt.get("architecture", "mobilenet_v3_small"),\n            "dataset": ckpt.get("dataset", "PAD-UFES-20"),\n            "balanced_accuracy": ckpt.get("balanced_accuracy"),\n            "training_mode": ckpt.get("training_mode"),\n            "temperature": self.temperature,\n            "selection_source": ckpt.get("selection_source", "combined"),\n            "calibration": ckpt.get("calibration"),\n        }\n        if self.meta["architecture"] != "mobilenet_v3_small":\n            raise ValueError("Expected a MobileNetV3-Small checkpoint produced by ml/train.py")\n        if set(self.class_names) != {"ACK", "BCC", "MEL", "NEV", "SCC", "SEK"} or len(self.class_names) != 6:\n            raise ValueError("Checkpoint must contain the six supported lesion classes")\n        self.model = mobilenet_v3_small(weights=None)\n        self.model.classifier[-1] = nn.Linear(\n            self.model.classifier[-1].in_features, len(self.class_names)\n        )\n        self.model.load_state_dict(ckpt["model_state"])\n        self.model.to(self.device).eval()\n\n        self.transform = A.Compose([\n            A.Resize(self.image_size, self.image_size),\n            A.Normalize(),\n            ToTensorV2(),\n        ])\n\n    @property\n    def ready(self):\n        return self.model is not None\n\n    @torch.no_grad()\n    def predict_bytes(self, raw: bytes):\n        if not self.ready:\n            raise RuntimeError("Model is not trained yet. Train it first with ml/train.py.")\n\n        image = Image.open(io.BytesIO(raw)).convert("RGB")\n        arr = np.asarray(image)\n        x = self.transform(image=arr)["image"].unsqueeze(0).to(self.device)\n        logits = self.model(x)\n        probs = torch.softmax(logits / self.temperature, dim=1)[0].cpu().numpy()\n\n        order = np.argsort(probs)[::-1]\n        ranked = [\n            {"label": self.class_names[i], "probability": round(float(probs[i]) * 100, 2)}\n            for i in order[:3]\n        ]\n        return {\n            "label": ranked[0]["label"],\n            "confidence": ranked[0]["probability"],\n            "top_predictions": ranked,\n        }\n', 'app/main.py': 'from fastapi import FastAPI, File, HTTPException, Request, UploadFile, Form, Depends, BackgroundTasks\nfrom fastapi.responses import HTMLResponse, JSONResponse, RedirectResponse, Response\nfrom fastapi.staticfiles import StaticFiles\nfrom fastapi.templating import Jinja2Templates\nfrom PIL import Image\nimport io\nfrom pathlib import Path\n\nfrom .inference import Predictor\nfrom .image_quality import validate_photo\nfrom . import auth, reporting\nimport secrets\nfrom datetime import datetime, timezone\nfrom .guidance import GUIDANCE, stage_info\n\napp = FastAPI(\n    title="SkinSight AI",\n    description="Educational skin-lesion screening using an own-trained MobileNetV3-Small model.",\n    version="0.1.0",\n)\n\nAPP_DIR = Path(__file__).resolve().parent\napp.mount("/static", StaticFiles(directory=str(APP_DIR / "static")), name="static")\ntemplates = Jinja2Templates(directory=str(APP_DIR / "templates"))\npredictor = Predictor()\n\n@app.get("/analyze", response_class=HTMLResponse)\n@app.get("/", response_class=HTMLResponse)\ndef home(request: Request):\n    user = auth.current_user(request)\n    if not user:\n        return RedirectResponse("/login", status_code=303)\n    csrf = request.cookies.get("skinsight_csrf") or secrets.token_urlsafe(32)\n    response = templates.TemplateResponse(\n        "index.html",\n        {"request": request, "model_ready": predictor.ready, "model_meta": predictor.meta, "username": user, "csrf": csrf, "email_profile": auth.email_profile(user), "page": "analyze", "year": datetime.now(timezone.utc).year},\n    )\n\n    response.set_cookie("skinsight_csrf", csrf, httponly=True, secure=auth.SECURE_COOKIE, samesite="strict")\n    response.headers["Cache-Control"] = "no-store"\n    return response\n\ndef auth_page(request, register=False, error=None, status=200):\n    csrf = request.cookies.get("skinsight_csrf") or secrets.token_urlsafe(32)\n    response = templates.TemplateResponse("login.html", {"request": request, "register": register, "error": error, "csrf": csrf, "username": None, "page": "auth", "year": datetime.now(timezone.utc).year}, status_code=status)\n    response.set_cookie("skinsight_csrf", csrf, httponly=True, secure=auth.SECURE_COOKIE, samesite="strict")\n    response.headers["Cache-Control"] = "no-store"\n    return response\n\n@app.get("/login", response_class=HTMLResponse)\ndef login_page(request: Request):\n    return auth_page(request)\n\n@app.get("/register", response_class=HTMLResponse)\ndef register_page(request: Request):\n    return auth_page(request, register=True)\n\n@app.post("/register")\ndef register_account(request: Request, username: str = Form(...), password: str = Form(...), csrf: str = Form(...), email: str = Form(...)):\n    auth.check_csrf(request, csrf)\n    try:\n        auth.register(username, password, email)\n    except ValueError as exc:\n        return auth_page(request, register=True, error=str(exc), status=400)\n    return RedirectResponse("/login", status_code=303)\n\n@app.post("/login")\ndef login_account(request: Request, username: str = Form(...), password: str = Form(...), csrf: str = Form(...)):\n    auth.check_csrf(request, csrf)\n    try:\n        token = auth.login(username, password, request.client.host if request.client else "unknown")\n    except ValueError as exc:\n        return auth_page(request, error=str(exc), status=400)\n    response = RedirectResponse("/", status_code=303)\n    response.set_cookie("skinsight_session", token, max_age=auth.SESSION_SECONDS, httponly=True, secure=auth.SECURE_COOKIE, samesite="strict")\n    return response\n\n@app.post("/logout")\ndef logout_account(request: Request, csrf: str = Form(...)):\n    auth.check_csrf(request, csrf)\n    auth.logout(request)\n    response = RedirectResponse("/login", status_code=303)\n    response.delete_cookie("skinsight_session")\n    return response\n\n@app.get("/api/health")\ndef health():\n    return {\n        "status": "ok",\n        "model_ready": predictor.ready,\n        "model": predictor.meta if predictor.ready else None,\n    }\n\n@app.post("/api/predict")\nasync def predict(request: Request, background_tasks: BackgroundTasks, image: UploadFile = File(...), user: str = Depends(auth.require_user)):\n    auth.check_csrf(request, request.headers.get("X-CSRF-Token"))\n    if not predictor.ready:\n        raise HTTPException(\n            status_code=503,\n            detail="The model has not been trained yet. Run ml/train.py first.",\n        )\n\n    if image.content_type not in {"image/jpeg", "image/png", "image/webp"}:\n        raise HTTPException(status_code=400, detail="Please upload a JPG, PNG, or WEBP image.")\n\n    raw = await image.read()\n    if len(raw) > 10 * 1024 * 1024:\n        raise HTTPException(status_code=413, detail="Image must be 10 MB or smaller.")\n\n    try:\n        im = Image.open(io.BytesIO(raw))\n        im.verify()\n    except Exception:\n        raise HTTPException(status_code=400, detail="The uploaded file is not a valid image.")\n\n    try:\n        validate_photo(Image.open(io.BytesIO(raw)))\n    except ValueError as exc:\n        raise HTTPException(status_code=422, detail=str(exc))\n\n    try:\n        result = predictor.predict_bytes(raw)\n    except Exception as exc:\n        raise HTTPException(status_code=500, detail=str(exc))\n\n    label = result["label"]\n    result["guidance"] = GUIDANCE[label]\n    result["stage"] = stage_info(label)\n    result["medical_disclaimer"] = (\n        "This is an educational AI screening result, not a medical diagnosis. "\n        "Model scores are not calibrated probabilities of disease. A clinician must assess concerning lesions."\n    )\n    report_id=reporting.save_report(user,result)\n    result["report_url"] = f"/api/reports/{report_id}/download"\n    result["email_status_url"] = f"/api/reports/{report_id}/status"\n    result["email_status"] = "pending"\n    background_tasks.add_task(reporting.deliver_report,user,report_id)\n    return JSONResponse(result,headers={"Cache-Control":"no-store"})\n\n\n@app.get(\'/api/reports/{report_id}/download\')\ndef download_report(report_id: str, user: str = Depends(auth.require_user)):\n    result=reporting.get_report(user,report_id)\n    if result is None:raise HTTPException(status_code=404,detail=\'Report not found.\')\n    return Response(reporting.make_pdf(reporting.report_text(result)),media_type=\'application/pdf\',headers={\'Content-Disposition\':\'attachment; filename="skinsight-screening-report.pdf"\',\'Cache-Control\':\'no-store\'})\n\n\n@app.get(\'/api/reports/{report_id}/status\')\ndef report_status(report_id: str, user: str = Depends(auth.require_user)):\n    result=reporting.get_report(user,report_id)\n    if result is None:raise HTTPException(status_code=404,detail=\'Report not found.\')\n    return JSONResponse({\'email_status\':result[\'email_status\']},headers={\'Cache-Control\':\'no-store\'})\n\n\ndef account_page(request,user,message=None):\n    response=templates.TemplateResponse(\'account.html\',{\'request\':request,\'profile\':auth.email_profile(user),\'csrf\':request.cookies.get(\'skinsight_csrf\'),\'message\':message,\'mail_ready\':reporting.smtp_ready(),\'username\':user,\'page\':\'account\',\'year\':datetime.now(timezone.utc).year})\n    response.headers[\'Cache-Control\']=\'no-store\'\n    return response\n\n\n@app.get(\'/account\',response_class=HTMLResponse)\ndef account(request: Request,user: str = Depends(auth.require_user)):\n    return account_page(request,user)\n\n\n@app.post(\'/account/email\')\ndef account_email(request: Request,email: str=Form(...),csrf: str=Form(...),user: str=Depends(auth.require_user)):\n    auth.check_csrf(request,csrf)\n    try:auth.set_email(user,email);message=\'Email saved. Future screening reports will be sent directly to this address.\'\n    except ValueError as exc:message=str(exc)\n    return account_page(request,user,message)\n\n\nFAQS = [\n    (\'Can SkinSight diagnose skin cancer?\', \'No. It ranks six image-model categories for educational screening. A clinician must assess a concerning lesion and determine whether a biopsy is needed.\'),\n    (\'What does the percentage mean?\', \'It is the classifier score for that image, not a calibrated probability of disease. A high score does not establish a diagnosis. Smartphone validation balanced accuracy was 56.9%; independent clinical testing is still needed.\'),\n    (\'Can it tell me the cancer stage?\', \'No. Staging needs a confirmed diagnosis and clinical information such as pathology, tumor thickness, ulceration and spread findings. The stage guide explains general concepts only.\'),\n    (\'How do I take a useful photo?\', \'Use a clear, close-up color image in natural light. Keep the lesion centered, avoid heavy filters and glare, and use JPG, PNG or WEBP under 10 MB.\'),\n    (\'Why was my image rejected?\', \'The app rejects tiny, blank, extremely exposed, grayscale or obvious flat-color graphic inputs. These conservative checks are not a complete skin detector and can reject some legitimate images.\'),\n    (\'Where are my images and reports stored?\', \'Uploaded photos are processed in memory and are not stored with your reports. Report summaries and accounts remain in the local SQLite database on the computer hosting this app. Different hosts have separate accounts.\'),\n    (\'How do report emails work?\', \'Successful analyses attempt to send a PDF and doctor guidance directly to your registered email. The sender Gmail account must be configured with a Google App Password. No email verification step is required. Downloads work even if sending fails.\'),\n    (\'What should I do about a changing lesion?\', \'Arrange clinical review if a lesion is new, changing, bleeding, painful, itching or looks unusual. Do not use a low model score or a benign-class prediction to rule out cancer.\'),\n]\n\n\ndef public_page(request, template, page, **extra):\n    user=auth.current_user(request)\n    csrf=request.cookies.get(\'skinsight_csrf\') or secrets.token_urlsafe(32)\n    context={\'request\':request,\'username\':user,\'csrf\':csrf,\'page\':page,\'year\':datetime.now(timezone.utc).year,**extra}\n    response=templates.TemplateResponse(template,context)\n    response.set_cookie(\'skinsight_csrf\',csrf,httponly=True,secure=auth.SECURE_COOKIE,samesite=\'strict\')\n    response.headers[\'Cache-Control\']=\'no-store\'\n    return response\n\n\n@app.get(\'/home\',response_class=HTMLResponse)\ndef landing(request: Request):\n    return public_page(request,\'home.html\',\'home\')\n\n\n@app.get(\'/about\',response_class=HTMLResponse)\ndef about(request: Request):\n    return public_page(request,\'about.html\',\'about\')\n\n\n@app.get(\'/faq\',response_class=HTMLResponse)\ndef faq(request: Request):\n    return public_page(request,\'faq.html\',\'faq\',faqs=FAQS)\n\n\n@app.get(\'/stages\',response_class=HTMLResponse)\ndef stages(request: Request):\n    return public_page(request,\'stages.html\',\'stages\')\n\n\n@app.get(\'/dashboard\',response_class=HTMLResponse)\ndef dashboard(request: Request):\n    user=auth.current_user(request)\n    if not user:return RedirectResponse(\'/login\',status_code=303)\n    return public_page(request,\'dashboard.html\',\'dashboard\',**reporting.dashboard_data(user))\n\n\n@app.get(\'/reports/{report_id}\',response_class=HTMLResponse)\ndef report_page(request: Request,report_id: str,user: str=Depends(auth.require_user)):\n    result=reporting.get_report(user,report_id)\n    if not result:raise HTTPException(status_code=404,detail=\'Report not found.\')\n    result[\'report_url\']=f\'/api/reports/{report_id}/download\'\n    result[\'email_status_url\']=f\'/api/reports/{report_id}/status\'\n    return public_page(request,\'report.html\',\'dashboard\',report=result)\n', 'app/image_quality.py': '"""Conservative photo-quality gate, not a trained skin/lesion detector."""\nimport numpy as np\nfrom PIL import Image\n\n\ndef validate_photo(image: Image.Image):\n    """Reject obvious unusable inputs before the six-class classifier runs."""\n    if min(image.size) < 96:\n        raise ValueError(\'Image is too small. Upload an original, clear close-up skin photo at least 96 pixels wide and high.\')\n    sample = image.convert(\'RGB\').copy()\n    sample.thumbnail((256, 256))\n    rgb = np.asarray(sample)\n    gray = rgb.astype(np.float32).mean(axis=2)\n    if float(gray.std()) < 3 or float((gray < 8).mean()) > .95 or float((gray > 247).mean()) > .95:\n        raise ValueError(\'Image is blank or poorly exposed. Take a clear skin photo in natural light.\')\n    # Flat-color graphics and initials avatars have few colors or large uniform areas.\n    # This is independent of skin tone and never uses classifier confidence.\n    colors, counts = np.unique(rgb.reshape(-1, 3), axis=0, return_counts=True)\n    if len(colors) < 64 or float(counts.max() / counts.sum()) > .50:\n        raise ValueError(\'This looks like an icon, text, or a graphic rather than a usable photo. Upload a clear close-up photo of the skin lesion.\')\n    if np.percentile(rgb.max(axis=2).astype(float) - rgb.min(axis=2), 95) < 4:\n        raise ValueError(\'Upload an original color skin photo; grayscale graphics and scans cannot be screened reliably.\')\n', 'app/auth.py': '"""Local accounts with salted password hashes and revocable server-side sessions."""\nfrom contextlib import contextmanager\nimport hashlib\nimport hmac\nimport os\nimport re\nimport secrets\nimport sqlite3\nimport time\nfrom pathlib import Path\nfrom urllib.parse import urlsplit\nfrom fastapi import HTTPException, Request\n\nDB_PATH = Path(os.environ.get(\'SKINSIGHT_AUTH_DB\', str(Path(__file__).resolve().parents[1] / \'data/accounts.sqlite3\')))\nSECURE_COOKIE = os.environ.get(\'SKINSIGHT_SECURE_COOKIES\') == \'1\'\nSESSION_SECONDS = 8 * 60 * 60\n\n\n@contextmanager\ndef connect():\n    DB_PATH.parent.mkdir(parents=True, exist_ok=True)\n    db = sqlite3.connect(DB_PATH, timeout=10)\n    db.row_factory = sqlite3.Row\n    db.execute(\'CREATE TABLE IF NOT EXISTS users (id INTEGER PRIMARY KEY, username TEXT UNIQUE NOT NULL, salt TEXT NOT NULL, password_hash TEXT NOT NULL)\')\n    columns={row[1] for row in db.execute(\'PRAGMA table_info(users)\')}\n    for name,definition in [(\'email\',\'TEXT\'),(\'email_verified\',\'INTEGER NOT NULL DEFAULT 0\'),(\'verification_hash\',\'TEXT\'),(\'verification_expires\',\'REAL\'),(\'verification_attempts\',\'INTEGER NOT NULL DEFAULT 0\')]:\n        if name not in columns:db.execute(f\'ALTER TABLE users ADD COLUMN {name} {definition}\')\n    db.execute(\'CREATE UNIQUE INDEX IF NOT EXISTS users_email_unique ON users(email) WHERE email IS NOT NULL\')\n    db.execute(\'CREATE TABLE IF NOT EXISTS reports (id TEXT PRIMARY KEY, username TEXT NOT NULL, payload TEXT NOT NULL, email_status TEXT NOT NULL)\')\n    db.execute(\'CREATE TABLE IF NOT EXISTS sessions (token_hash TEXT PRIMARY KEY, user_id INTEGER NOT NULL, expires REAL NOT NULL)\')\n    db.execute(\'CREATE TABLE IF NOT EXISTS attempts (address TEXT PRIMARY KEY, failures INTEGER NOT NULL, started REAL NOT NULL)\')\n    db.commit()\n    try:\n        with db:\n            yield db\n    finally:\n        db.close()\n\n\ndef password_hash(password, salt):\n    return hashlib.pbkdf2_hmac(\'sha256\', password.encode(), bytes.fromhex(salt), 600000).hex()\n\n\ndef register(username, password, email=None):\n    email=validate_email(email) if email else None\n    username = username.strip().lower()\n    if not re.fullmatch(r\'[a-z0-9_]{3,32}\', username):\n        raise ValueError(\'Username must have 3–32 letters, numbers or underscores.\')\n    if not 12 <= len(password) <= 128:\n        raise ValueError(\'Use a password with 12–128 characters.\')\n    salt = secrets.token_hex(16)\n    hashed = password_hash(password, salt)\n    try:\n        with connect() as db:\n            db.execute(\'INSERT INTO users(username,salt,password_hash,email) VALUES(?,?,?,?)\', (username,salt,hashed,email))\n    except sqlite3.IntegrityError:\n        raise ValueError(\'That username or email is already registered.\')\n\n\ndef login(username, password, address):\n    now = time.time()\n    with connect() as db:\n        attempt = db.execute(\'SELECT * FROM attempts WHERE address=?\',(address,)).fetchone()\n        if attempt and now - attempt[\'started\'] < 900 and attempt[\'failures\'] >= 10:\n            raise ValueError(\'Too many failed attempts. Try again in 15 minutes.\')\n        user = db.execute(\'SELECT * FROM users WHERE username=? OR email=?\',(username.strip().lower(),username.strip().lower())).fetchone()\n        valid = hmac.compare_digest(password_hash(password[:129],user[\'salt\'] if user else \'00\'*16),user[\'password_hash\'] if user else \'0\'*64)\n        if not user or not valid or len(password)>128:\n            failures = attempt[\'failures\']+1 if attempt and now-attempt[\'started\']<900 else 1\n            started = attempt[\'started\'] if attempt and now-attempt[\'started\']<900 else now\n            db.execute(\'INSERT OR REPLACE INTO attempts VALUES(?,?,?)\',(address,failures,started))\n            raise_after = True\n        else:\n            db.execute(\'DELETE FROM attempts WHERE address=?\',(address,))\n            db.execute(\'DELETE FROM sessions WHERE expires < ?\',(now,))\n            token = secrets.token_urlsafe(32)\n            db.execute(\'INSERT INTO sessions VALUES(?,?,?)\',(hashlib.sha256(token.encode()).hexdigest(),user[\'id\'],now+SESSION_SECONDS))\n            raise_after = False\n    if raise_after:\n        raise ValueError(\'Incorrect username or password.\')\n    return token\n\n\ndef current_user(request: Request):\n    token = request.cookies.get(\'skinsight_session\')\n    if not token:\n        return None\n    with connect() as db:\n        user = db.execute(\'SELECT users.username FROM sessions JOIN users ON users.id=sessions.user_id WHERE token_hash=? AND expires>?\', (hashlib.sha256(token.encode()).hexdigest(),time.time())).fetchone()\n    return user[\'username\'] if user else None\n\n\ndef require_user(request: Request):\n    user = current_user(request)\n    if not user:\n        raise HTTPException(status_code=401,detail=\'Please log in before analyzing an image.\')\n    return user\n\n\ndef check_csrf(request, token):\n    expected = request.cookies.get(\'skinsight_csrf\',\'\')\n    origin = request.headers.get(\'origin\')\n    if (origin and urlsplit(origin).netloc != request.headers.get(\'host\')) or not expected or not token or not hmac.compare_digest(expected,token):\n        raise HTTPException(status_code=403,detail=\'Session verification failed. Refresh the page and try again.\')\n\n\ndef logout(request):\n    token = request.cookies.get(\'skinsight_session\',\'\')\n    with connect() as db:\n        db.execute(\'DELETE FROM sessions WHERE token_hash=?\',(hashlib.sha256(token.encode()).hexdigest(),))\n\n\ndef validate_email(email):\n    email=email.strip().lower()\n    if len(email)>254 or not re.fullmatch(r"[a-z0-9.!#$%&\'*+/=?^_`{|}~-]+@[a-z0-9](?:[a-z0-9.-]*[a-z0-9])?\\.[a-z]{2,}",email):\n        raise ValueError(\'Enter a valid email address.\')\n    return email\n\n\ndef email_profile(username):\n    with connect() as db:\n        row=db.execute(\'SELECT email,email_verified FROM users WHERE username=?\',(username,)).fetchone()\n    return dict(row)\n\n\ndef set_email(username,email):\n    email=validate_email(email)\n    try:\n        with connect() as db:\n            db.execute(\'UPDATE users SET email=?,email_verified=0,verification_hash=NULL,verification_expires=NULL,verification_attempts=0 WHERE username=?\',(email,username))\n    except sqlite3.IntegrityError:raise ValueError(\'That email is already registered.\')\n', 'app/reporting.py': '"""Owner-only screening reports and direct TLS SMTP delivery."""\nimport json\nimport os\nimport secrets\nimport smtplib\nimport ssl\nfrom datetime import datetime, timezone\nfrom email.message import EmailMessage\nfrom textwrap import wrap\nfrom . import auth\n\n\ndef smtp_ready():\n    return all(os.environ.get(k) for k in [\'SKINSIGHT_SMTP_HOST\',\'SKINSIGHT_SMTP_USER\',\'SKINSIGHT_SMTP_PASSWORD\',\'SKINSIGHT_MAIL_FROM\'])\n\n\ndef send_mail(recipient, subject, body, attachment=None):\n    if not smtp_ready():\n        return \'not_configured\'\n    message=EmailMessage();message[\'From\']=os.environ[\'SKINSIGHT_MAIL_FROM\'];message[\'To\']=recipient;message[\'Subject\']=subject\n    message.set_content(body)\n    if attachment:message.add_attachment(attachment,maintype=\'application\',subtype=\'pdf\',filename=\'skinsight-screening-report.pdf\')\n    try:\n        host=os.environ[\'SKINSIGHT_SMTP_HOST\'];port=int(os.environ.get(\'SKINSIGHT_SMTP_PORT\',\'587\'))\n        context=ssl.create_default_context()\n        if port==465:\n            connection=smtplib.SMTP_SSL(host,port,timeout=8,context=context)\n        else:\n            connection=smtplib.SMTP(host,port,timeout=8);connection.ehlo();connection.starttls(context=context);connection.ehlo()\n        with connection as server:\n            server.login(os.environ[\'SKINSIGHT_SMTP_USER\'],os.environ[\'SKINSIGHT_SMTP_PASSWORD\'])\n            refused=server.send_message(message)\n        return \'failed\' if refused else \'accepted\'\n    except smtplib.SMTPAuthenticationError:\n        return \'auth_failed\'\n    except smtplib.SMTPRecipientsRefused:\n        return \'recipient_refused\'\n    except smtplib.SMTPSenderRefused:\n        return \'sender_refused\'\n    except ssl.SSLError:\n        return \'tls_failed\'\n    except (OSError,smtplib.SMTPException,ValueError):\n        return \'failed\'\n\n\ndef report_text(result):\n    return \'\\n\'.join([\n        \'SkinSight AI - Educational screening report\',\n        \'Report ID: \'+result[\'report_id\'], \'Created (UTC): \'+result[\'created_at\'],\n        \'\', \'Image-model top class: \'+result[\'guidance\'][\'name\'],\n        f"Model score: {result[\'confidence\']:.1f}% (not a probability of disease)",\n        \'Class scores: \'+\', \'.join(f"{p[\'label\']} {p[\'probability\']:.1f}%" for p in result[\'top_predictions\']),\n        \'\', \'Doctor guidance:\',result[\'guidance\'][\'message\'],result[\'guidance\'][\'urgency\'],\n        \'Consult a qualified dermatologist. Bring this report and describe when the lesion appeared and any changes.\',\n        \'\', \'Cancer stage: \'+result[\'stage\'][\'status\'],result[\'stage\'][\'explanation\'],\n        *result[\'stage\'].get(\'education\',[]), \'\', result[\'medical_disclaimer\'],\n        \'Smartphone validation balanced accuracy: 56.9%; independent clinical validation is pending.\',\n    ])\n\n\ndef make_pdf(text):\n    """Small portable text-only PDF; no user HTML, uploads or external renderer."""\n    lines=[]\n    for line in text.splitlines():lines.extend(wrap(line,92) or [\'\'])\n    objects=[]\n    def add(data):objects.append(data);return len(objects)\n    add(b\'\');add(b\'\');font=add(b\'<< /Type /Font /Subtype /Type1 /BaseFont /Helvetica >>\');pages=[]\n    for offset in range(0,len(lines),47):\n        commands=[\'BT /F1 10 Tf 48 790 Td 15 TL\']\n        for line in lines[offset:offset+47]:\n            safe=line.replace(\'–\',\'-\').replace(\'—\',\'-\').replace(\'’\',"\'").encode(\'ascii\',\'replace\').decode().replace(\'\\\\\',\'\\\\\\\\\').replace(\'(\',\'\\\\(\').replace(\')\',\'\\\\)\')\n            commands.append(\'(\'+safe+\') Tj T*\')\n        commands.append(\'ET\');stream=\'\\n\'.join(commands).encode()\n        content=add(b\'<< /Length \'+str(len(stream)).encode()+b\' >>\\nstream\\n\'+stream+b\'\\nendstream\')\n        page=add(f\'<< /Type /Page /Parent 2 0 R /MediaBox [0 0 595 842] /Resources << /Font << /F1 {font} 0 R >> >> /Contents {content} 0 R >>\'.encode());pages.append(page)\n    objects[0]=b\'<< /Type /Catalog /Pages 2 0 R >>\';objects[1]=f"<< /Type /Pages /Count {len(pages)} /Kids [{\' \'.join(str(p)+\' 0 R\' for p in pages)}] >>".encode()\n    output=bytearray(b\'%PDF-1.4\\n\');positions=[0]\n    for i,obj in enumerate(objects,1):positions.append(len(output));output.extend(f\'{i} 0 obj\\n\'.encode()+obj+b\'\\nendobj\\n\')\n    xref=len(output);output.extend(f\'xref\\n0 {len(objects)+1}\\n0000000000 65535 f \\n\'.encode())\n    for p in positions[1:]:output.extend(f\'{p:010d} 00000 n \\n\'.encode())\n    output.extend(f\'trailer\\n<< /Size {len(objects)+1} /Root 1 0 R >>\\nstartxref\\n{xref}\\n%%EOF\\n\'.encode());return bytes(output)\n\n\ndef save_report(username,result):\n    result[\'report_id\']=secrets.token_urlsafe(18);result[\'created_at\']=datetime.now(timezone.utc).isoformat()\n    with auth.connect() as db:\n        db.execute(\'INSERT INTO reports VALUES(?,?,?,?)\',(result[\'report_id\'],username,json.dumps(result),\'pending\'))\n    return result[\'report_id\']\n\n\ndef get_report(username,report_id):\n    with auth.connect() as db:\n        row=db.execute(\'SELECT payload,email_status FROM reports WHERE id=? AND username=?\',(report_id,username)).fetchone()\n    if not row:return None\n    data=json.loads(row[\'payload\']);data[\'email_status\']=row[\'email_status\'];return data\n\n\ndef deliver_report(username,report_id):\n    result=get_report(username,report_id)\n    with auth.connect() as db:user=db.execute(\'SELECT email FROM users WHERE username=?\',(username,)).fetchone()\n    if not user or not user[\'email\']:status=\'email_required\'\n    else:status=send_mail(user[\'email\'],\'Your SkinSight screening report\',report_text(result),make_pdf(report_text(result)))\n    with auth.connect() as db:db.execute(\'UPDATE reports SET email_status=? WHERE id=? AND username=?\',(status,report_id,username))\n\n\ndef delivery_message(status):\n    return {\n        \'auth_failed\': \'Gmail rejected the sender login. Restart start_with_gmail.bat and enter the Gmail address that created the App Password, followed by that 16-character App Password (not your normal password).\',\n        \'recipient_refused\': \'Gmail rejected the recipient address. Check the Report email address.\',\n        \'sender_refused\': \'Gmail rejected the sender address. Use the same Gmail address for sender and login.\',\n        \'tls_failed\': \'The secure Gmail connection failed. Check your computer date/time and network; TLS verification remains enabled.\',\n        \'not_configured\': \'Gmail sending is not configured. Start the app with start_with_gmail.bat.\',\n        \'failed\': \'Could not send through Gmail. Check internet access and whether the network permits smtp.gmail.com on port 465, then try again.\',\n    }.get(status, \'Unable to send the email. Your report can still be downloaded.\')\n\n\ndef dashboard_data(username):\n    """Aggregate only the signed-in account\'s stored results, never example data."""\n    from collections import Counter\n    with auth.connect() as db:\n        rows=db.execute(\'SELECT payload,email_status FROM reports WHERE username=? ORDER BY rowid DESC\',(username,)).fetchall()\n    reports=[]\n    descriptions={\'accepted\':\'Accepted by mail server\',\'pending\':\'Processing\',\'failed\':\'Sending failed\',\'not_configured\':\'Sender not configured\',\'email_required\':\'Add email\',\'auth_failed\':\'Sender login rejected\',\'recipient_refused\':\'Recipient rejected\',\'sender_refused\':\'Sender rejected\',\'tls_failed\':\'Secure connection failed\'}\n    for row in rows:\n        data=json.loads(row[\'payload\'])\n        created=datetime.fromisoformat(data[\'created_at\']).astimezone(timezone.utc)\n        data[\'display_date\']=created.strftime(\'%d %b %Y · %H:%M\')\n        data[\'email_description\']=descriptions.get(row[\'email_status\'],\'Not sent\')\n        reports.append(data)\n    now=datetime.now(timezone.utc)\n    monthly=Counter(datetime.fromisoformat(r[\'created_at\']).astimezone(timezone.utc).strftime(\'%Y-%m\') for r in reports)\n    month_items=[]\n    for offset in range(5,-1,-1):\n        absolute=now.year*12+now.month-1-offset\n        month_date=datetime(absolute//12,absolute%12+1,1,tzinfo=timezone.utc)\n        key=month_date.strftime(\'%Y-%m\')\n        month_items.append({\'label\':month_date.strftime(\'%b %y\'),\'count\':monthly[key]})\n    maximum=max([item[\'count\'] for item in month_items]+[1])\n    for item in month_items:item[\'height\']=round(item[\'count\']/maximum*100,2)\n    frequencies=Counter(r[\'label\'] for r in reports)\n    return {\'reports\':reports,\'stats\':{\'total\':len(reports),\n        \'today\':sum(datetime.fromisoformat(r[\'created_at\']).astimezone(timezone.utc).date()==now.date() for r in reports),\n        \'average\':sum(r[\'confidence\'] for r in reports)/len(reports) if reports else 0,\n        \'cancer_classes\':sum(r[\'label\'] in {\'MEL\',\'BCC\',\'SCC\'} for r in reports)},\n        \'months\':month_items,\'frequencies\':[{\'label\':label,\'count\':frequencies[label],\'percent\':round(frequencies[label]/len(reports)*100,2) if reports else 0} for label in [\'ACK\',\'BCC\',\'MEL\',\'NEV\',\'SCC\',\'SEK\']]}\n', 'ml/dataset.py': 'from pathlib import Path\nimport pandas as pd\nfrom PIL import Image\nfrom torch.utils.data import Dataset\n\nCLASS_NAMES = ["ACK", "BCC", "MEL", "NEV", "SCC", "SEK"]\nCLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}\n\nclass ManifestDataset(Dataset):\n    """\n    Dataset backed by data/combined_manifest.csv.\n\n    Required columns:\n      image_path,label,patient_id,source,original_id\n    """\n    def __init__(self, frame, transform=None):\n        self.df = frame.reset_index(drop=True)\n        self.transform = transform\n\n    def __len__(self):\n        return len(self.df)\n\n    def __getitem__(self, idx):\n        row = self.df.iloc[idx]\n        path = Path(row["image_path"])\n        image = Image.open(path).convert("RGB")\n        label = str(row["label"]).strip().upper()\n        if label not in CLASS_TO_IDX:\n            raise ValueError(f"Unsupported label: {label}")\n        y = CLASS_TO_IDX[label]\n\n        if self.transform:\n            import numpy as np\n            image = self.transform(image=np.asarray(image))["image"]\n\n        return image, y\n', 'ml/evaluation.py': '"""Fixed-label metrics including sensitivity, specificity, and calibration error."""\nimport numpy as np\nfrom sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix, roc_auc_score\nfrom dataset import CLASS_NAMES\n\n\ndef summarize(labels, probabilities):\n    labels = np.asarray(labels, dtype=int)\n    probabilities = np.asarray(probabilities, dtype=float)\n    if not len(labels):\n        raise ValueError(\'Cannot evaluate an empty dataset\')\n    predictions = probabilities.argmax(axis=1)\n    matrix = confusion_matrix(labels, predictions, labels=range(len(CLASS_NAMES)))\n    total = int(matrix.sum())\n    per_class = {}\n    for i, name in enumerate(CLASS_NAMES):\n        tp = int(matrix[i, i]); fn = int(matrix[i].sum()) - tp\n        fp = int(matrix[:, i].sum()) - tp; tn = total - tp - fn - fp\n        binary = labels == i\n        per_class[name] = {\n            \'support\': tp + fn,\n            \'sensitivity\': tp / (tp + fn) if tp + fn else None,\n            \'specificity\': tn / (tn + fp) if tn + fp else None,\n            \'roc_auc\': float(roc_auc_score(binary, probabilities[:, i])) if binary.any() and (~binary).any() else None,\n        }\n    confidence = probabilities.max(axis=1)\n    correctness = predictions == labels\n    ece = 0.0\n    for low, high in zip(np.linspace(0, 1, 11)[:-1], np.linspace(0, 1, 11)[1:]):\n        mask = (confidence >= low) & ((confidence < high) if high < 1 else (confidence <= high))\n        if mask.any():\n            ece += float(mask.mean() * abs(correctness[mask].mean() - confidence[mask].mean()))\n    return {\n        \'sample_count\': len(labels),\n        \'balanced_accuracy\': float(balanced_accuracy_score(labels, predictions)),\n        \'classification_report\': classification_report(labels, predictions, labels=range(len(CLASS_NAMES)),\n                                                      target_names=CLASS_NAMES, zero_division=0, output_dict=True),\n        \'confusion_matrix\': matrix.tolist(), \'class_order\': CLASS_NAMES,\n        \'per_class\': per_class, \'expected_calibration_error_10_bins\': ece,\n        \'calibration_note\': \'Softmax probabilities are not clinically calibrated confidence estimates.\',\n    }\n', 'ml/model.py': 'import torch.nn as nn\nfrom torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights\n\nNUM_CLASSES = 6\nCLASS_NAMES = ["ACK","BCC","MEL","NEV","SCC","SEK"]\n\ndef create_model(num_classes=NUM_CLASSES, pretrained=True, freeze_backbone=True):\n    weights = MobileNet_V3_Small_Weights.DEFAULT if pretrained else None\n    model = mobilenet_v3_small(weights=weights)\n    if freeze_backbone:\n        for p in model.features.parameters():\n            p.requires_grad = False\n    model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, num_classes)\n    return model\n', 'ml/prepare_combined.py': 'import argparse\nimport hashlib\nfrom pathlib import Path\nimport pandas as pd\n\nTARGET_CLASSES = ["ACK", "BCC", "MEL", "NEV", "SCC", "SEK"]\n\nPAD_ALIASES = {\n    "ACK": "ACK",\n    "AK": "ACK",\n    "ACTINIC KERATOSIS": "ACK",\n    "BCC": "BCC",\n    "BASAL CELL CARCINOMA": "BCC",\n    "MEL": "MEL",\n    "MELANOMA": "MEL",\n    "NEV": "NEV",\n    "NEVUS": "NEV",\n    "SCC": "SCC",\n    "SQUAMOUS CELL CARCINOMA": "SCC",\n    "SEK": "SEK",\n    "SK": "SEK",\n    "SEBORRHEIC KERATOSIS": "SEK",\n}\n\nISIC_MAP = {\n    "MEL": "MEL",\n    "NV": "NEV",\n    "BCC": "BCC",\n    "AK": "ACK",\n    "BKL": "SEK",\n    "SCC": "SCC",\n}\n\nMANIFEST_COLUMNS = ["image_path", "label", "patient_id", "lesion_id", "grouping_level", "image_sha256", "source", "original_id"]\n\ndef digest(path):\n    with path.open("rb") as file:\n        return hashlib.file_digest(file, "sha256").hexdigest()\n\nVALID_EXTS = {".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"}\n\ndef find_column(df, candidates, required=True):\n    lookup = {str(c).strip().lower(): c for c in df.columns}\n    for c in candidates:\n        if c.lower() in lookup:\n            return lookup[c.lower()]\n    if required:\n        raise ValueError(f"Could not find any of {candidates}. Found columns: {list(df.columns)}")\n    return None\n\ndef build_image_index(root: Path):\n    if not root.is_dir():\n        raise FileNotFoundError(f"Image directory does not exist: {root}")\n    index = {}\n    for p in root.rglob("*"):\n        if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png"}:\n            if p.stem in index and index[p.stem] != p.resolve():\n                raise ValueError(f"Ambiguous image ID: {p.stem}")\n            index.setdefault(p.stem, p.resolve())\n            index.setdefault(p.name, p.resolve())\n    return index\n\ndef normalize_pad(v):\n    return PAD_ALIASES.get(str(v).strip().upper())\n\ndef build_pad_manifest(metadata_csv: Path, pad_root: Path):\n    df = pd.read_csv(metadata_csv)\n    image_col = find_column(df, ["img_id", "image", "image_id", "filename", "file_name"])\n    label_col = find_column(df, ["diagnostic", "diagnosis", "label", "class"])\n    patient_col = find_column(df, ["patient_id", "patient", "patientid"], required=False)\n\n    lesion_col = find_column(df, ["lesion_id"], required=False)\n    if df[image_col].duplicated().any():\n        raise ValueError("PAD metadata has duplicate image IDs")\n    image_index = build_image_index(pad_root)\n    rows = []\n    missing = 0\n\n    for _, row in df.iterrows():\n        label = normalize_pad(row[label_col])\n        if label not in TARGET_CLASSES:\n            continue\n\n        raw = str(row[image_col]).strip()\n        path = image_index.get(raw) or image_index.get(Path(raw).stem)\n\n        if path is None:\n            missing += 1\n            continue\n\n        patient_id = ""\n        if patient_col and pd.notna(row[patient_col]):\n            patient_id = f"PAD::{row[patient_col]}"\n\n        rows.append({\n            "image_path": str(path),\n            "label": label,\n            "patient_id": patient_id,\n            "lesion_id": str(row[lesion_col]) if lesion_col and pd.notna(row[lesion_col]) else "",\n            "grouping_level": "patient" if patient_id else ("lesion" if lesion_col and pd.notna(row[lesion_col]) else "image"),\n            "image_sha256": digest(path),\n            "source": "PAD-UFES-20",\n            "original_id": Path(raw).stem,\n        })\n\n    if missing:\n        raise ValueError(f"{missing} supported images are missing; check the image directory")\n    out = pd.DataFrame(rows, columns=MANIFEST_COLUMNS)\n    if out.empty:\n        raise ValueError("No supported images matched the metadata")\n    print(f"PAD matched images: {len(out)} | missing: {missing}")\n    return out\n\ndef build_isic_manifest(gt_csv: Path, image_root: Path, metadata_csv=None):\n    gt = pd.read_csv(gt_csv)\n    image_col = find_column(gt, ["image", "image_id", "isic_id"])\n    if gt[image_col].duplicated().any():\n        raise ValueError("ISIC ground truth has duplicate image IDs")\n    label_columns = [name for name in [*ISIC_MAP, "DF", "VASC", "UNK"] if name in gt]\n    if not label_columns:\n        raise ValueError("ISIC ground truth has no recognized class columns")\n    numeric = gt[label_columns].apply(pd.to_numeric, errors="raise")\n    if not numeric.isin([0, 1]).all().all() or not numeric.sum(axis=1).eq(1).all():\n        raise ValueError("ISIC labels must be valid one-hot rows")\n    metadata = {}\n    if metadata_csv:\n        meta = pd.read_csv(metadata_csv)\n        meta_image = find_column(meta, ["image", "image_id", "isic_id"])\n        if meta[meta_image].duplicated().any():\n            raise ValueError("ISIC metadata has duplicate image IDs")\n        metadata = {str(row[meta_image]): row for _, row in meta.iterrows()}\n    image_index = build_image_index(image_root)\n\n    rows = []\n    missing = 0\n    skipped = 0\n\n    for _, row in gt.iterrows():\n        positives = []\n        for src_label in ISIC_MAP:\n            if src_label in gt.columns:\n                try:\n                    if float(row[src_label]) >= 0.5:\n                        positives.append(src_label)\n                except Exception:\n                    pass\n\n        if len(positives) != 1:\n            skipped += 1\n            continue\n\n        src_label = positives[0]\n        final_label = ISIC_MAP[src_label]\n        image_id = str(row[image_col]).strip()\n        path = image_index.get(image_id) or image_index.get(Path(image_id).stem)\n\n        if path is None:\n            missing += 1\n            continue\n\n        meta = metadata.get(image_id, {})\n        patient = meta.get("patient_id", "")\n        patient = str(patient).strip() if pd.notna(patient) else ""\n        if patient.lower().startswith("dummy_"):\n            patient = ""\n        lesion = meta.get("lesion_id", "")\n        lesion = str(lesion).strip() if pd.notna(lesion) else ""\n        rows.append({\n            "image_path": str(path),\n            "label": final_label,\n            "patient_id": patient,\n            "lesion_id": lesion,\n            "grouping_level": "patient" if patient else ("lesion" if lesion else "image"),\n            "image_sha256": digest(path),\n            "source": "ISIC-2019",\n            "original_id": Path(image_id).stem,\n        })\n\n    if missing:\n        raise ValueError(f"{missing} supported images are missing; check the image directory")\n    out = pd.DataFrame(rows, columns=MANIFEST_COLUMNS)\n    if out.empty:\n        raise ValueError("No supported images matched the metadata")\n    print(f"ISIC matched images: {len(out)} | missing: {missing} | unsupported/skipped: {skipped}")\n    return out\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument("--pad-meta", required=True)\n    p.add_argument("--pad-root", required=True)\n    p.add_argument("--isic-gt", required=True)\n    p.add_argument("--isic-root", required=True)\n    p.add_argument("--isic-meta", help="Optional authoritative ISIC metadata with lesion/patient identifiers")\n    p.add_argument("--out", default="data/combined_manifest.csv")\n    args = p.parse_args()\n\n    pad = build_pad_manifest(Path(args.pad_meta), Path(args.pad_root))\n    isic = build_isic_manifest(Path(args.isic_gt), Path(args.isic_root), Path(args.isic_meta) if args.isic_meta else None)\n\n    combined = pd.concat([pad, isic], ignore_index=True)\n    combined = combined.drop_duplicates(subset=["source", "original_id"]).reset_index(drop=True)\n\n    print("\\nGrouping coverage:"); print(combined.groupby(["source", "grouping_level"]).size())\n    if (combined.grouping_level == "image").any():\n        print("WARNING: image-only groups cannot guarantee patient/lesion separation.")\n    out = Path(args.out)\n    out.parent.mkdir(parents=True, exist_ok=True)\n    combined.to_csv(out, index=False)\n\n    print("\\n=== FINAL COMBINED DATASET ===")\n    print(combined.groupby(["source", "label"]).size().unstack(fill_value=0))\n    print("\\nClass totals:")\n    print(combined["label"].value_counts().reindex(TARGET_CLASSES, fill_value=0))\n    print("\\nGrand total:", len(combined))\n    print("\\nSaved manifest:", out)\n\nif __name__ == "__main__":\n    main()\n', 'ml/splits.py': '"""Deterministic source/class-stratified splits with linked identity groups."""\nimport pandas as pd\nfrom sklearn.model_selection import StratifiedGroupKFold\n\n\ndef present(value):\n    return pd.notna(value) and str(value).strip().lower() not in {\'\', \'nan\', \'none\'}\n\n\ndef linked_groups(frame):\n    """Union patient/lesion/image identities and exact content duplicates transitively."""\n    parents = list(range(len(frame)))\n    def find(i):\n        while parents[i] != i:\n            parents[i] = parents[parents[i]]\n            i = parents[i]\n        return i\n    seen = {}\n    for i, row in enumerate(frame.to_dict(\'records\')):\n        source = str(row[\'source\'])\n        keys = [(\'image\', source, str(row[\'original_id\']))]\n        for column in [\'patient_id\', \'lesion_id\']:\n            value = row.get(column)\n            if present(value):\n                keys.append((column, source, str(value).strip()))\n        if present(row.get(\'image_sha256\')):\n            keys.append((\'sha256\', str(row[\'image_sha256\'])))\n        for key in keys:\n            if key in seen:\n                parents[find(i)] = find(seen[key])\n            else:\n                seen[key] = i\n    return pd.Series([f\'group::{find(i)}\' for i in range(len(frame))], index=frame.index)\n\n\ndef split_three_way(frame):\n    """Approx. 60/20/20; fixed folds, no search using model scores or test outcomes."""\n    frame = frame.copy().reset_index(drop=True)\n    frame[\'_group\'] = linked_groups(frame)\n    strata = frame.source.astype(str) + \'::\' + frame.label.astype(str)\n    splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)\n    folds = list(splitter.split(frame, strata, groups=frame[\'_group\']))\n    validation_ids = folds[0][1]\n    test_ids = folds[1][1]\n    train_mask = ~frame.index.isin(list(validation_ids) + list(test_ids))\n    train = frame.loc[train_mask].copy()\n    validation = frame.iloc[validation_ids].copy()\n    test = frame.iloc[test_ids].copy()\n    parts = {\'train\': train, \'validation\': validation, \'test\': test}\n    groups = [set(part[\'_group\']) for part in parts.values()]\n    if any(groups[i] & groups[j] for i in range(3) for j in range(i)):\n        raise RuntimeError(\'Linked identity group leakage detected\')\n    # Fail instead of silently evaluating a source/class missing from a split.\n    required = set(strata)\n    for name, part in parts.items():\n        actual = set(part.source.astype(str) + \'::\' + part.label.astype(str))\n        if required - actual:\n            raise ValueError(f\'{name} lacks source/class groups: {sorted(required-actual)}. \'\n                             \'More independent groups or an explicitly designed split are needed.\')\n    return train, validation, test\n', 'ml/train.py': '"""Train on training data, select on validation, evaluate held-out test once."""\nimport argparse\nimport hashlib\nimport json\nimport random\nimport time\nfrom pathlib import Path\nimport albumentations as A\nfrom albumentations.pytorch import ToTensorV2\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch import nn\nfrom torch.utils.data import DataLoader, WeightedRandomSampler\nfrom sklearn.model_selection import GroupShuffleSplit\nfrom dataset import ManifestDataset, CLASS_NAMES\nfrom model import create_model\nfrom splits import linked_groups, split_three_way\nfrom evaluation import summarize\nfrom features import cache_features\n\n\ndef seed_all(seed=42):\n    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)\n\n\ndef tfms(train, size):\n    items = [A.Resize(size, size)]\n    if train:\n        items += [A.HorizontalFlip(p=.5), A.Rotate(limit=20, p=.3),\n                  A.RandomBrightnessContrast(p=.25), A.HueSaturationValue(p=.15)]\n    return A.Compose(items + [A.Normalize(), ToTensorV2()])\n\n\ndef split_df(df):\n    """Compatibility helper for callers needing only a train/validation partition."""\n    df = df[df.label.isin(CLASS_NAMES)].copy().reset_index(drop=True)\n    if df.empty:\n        raise ValueError(\'Manifest contains no supported lesion classes\')\n    df[\'_group\'] = linked_groups(df)\n    a, b = next(GroupShuffleSplit(n_splits=1, test_size=.2, random_state=42).split(df, groups=df[\'_group\']))\n    return df.iloc[a].copy(), df.iloc[b].copy()\n\n\ndef sampler(df):\n    keys = df.source.astype(str) + \'::\' + df.label.astype(str)\n    counts = keys.value_counts().to_dict()\n    return WeightedRandomSampler([1 / counts[key] for key in keys], len(keys), replacement=True)\n\n\n@torch.no_grad()\ndef collect(model, loader, device):\n    model.eval(); labels = []; probabilities = []; loss_sum = 0.\n    for x, y in loader:\n        x, y = x.to(device), y.to(device)\n        logits = model(x)\n        loss_sum += float(nn.functional.cross_entropy(logits, y)) * len(y)\n        labels.extend(y.cpu().tolist())\n        probabilities.extend(torch.softmax(logits, 1).cpu().tolist())\n    return loss_sum / len(loader.dataset), labels, probabilities\n\n\ndef evaluate(model, loader, device):\n    loss, labels, probabilities = collect(model, loader, device)\n    report = summarize(labels, probabilities)\n    return loss, report[\'balanced_accuracy\'], report[\'classification_report\']\n\n\ndef report_frame(model, frame, size, batch_size, workers, device, dataset=None):\n    loader = DataLoader(dataset if dataset is not None else ManifestDataset(frame, tfms(False, size)), batch_size=batch_size,\n                        shuffle=False, num_workers=workers)\n    loss, labels, probabilities = collect(model, loader, device)\n    report = summarize(labels, probabilities); report[\'loss\'] = loss\n    report[\'by_source\'] = {}\n    sources = frame.source.to_numpy()\n    for source in sorted(frame.source.unique()):\n        selected = np.flatnonzero(sources == source)\n        report[\'by_source\'][source] = summarize(np.asarray(labels)[selected], np.asarray(probabilities)[selected])\n    return report\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument(\'--manifest\', default=\'data/combined_manifest.csv\')\n    p.add_argument(\'--epochs\', type=int, default=10)\n    p.add_argument(\'--batch-size\', type=int, default=16)\n    p.add_argument(\'--workers\', type=int, default=0)\n    p.add_argument(\'--size\', type=int, default=160)\n    p.add_argument(\'--lr\', type=float, default=1e-3)\n    p.add_argument(\'--output-dir\', default=\'models\')\n    p.add_argument(\'--from-scratch\', action=\'store_true\')\n    p.add_argument(\'--fine-tune\', action=\'store_true\')\n    p.add_argument(\'--init-checkpoint\', help=\'Trusted compatible weights for warm-start fine-tuning\')\n    p.add_argument(\'--skip-test\', action=\'store_true\', help=\'Do not access the previously reported test partition\')\n    p.add_argument(\'--train-source\', choices=[\'combined\',\'PAD-UFES-20\',\'ISIC-2019\'], default=\'combined\')\n    p.add_argument(\'--selection-source\', choices=[\'combined\',\'PAD-UFES-20\'], default=\'combined\')\n    p.add_argument(\'--cache-features\', action=\'store_true\', help=\'Fast frozen-backbone baseline; uses deterministic preprocessing without train augmentation\')\n    p.add_argument(\'--device\', choices=[\'auto\', \'cpu\', \'cuda\'], default=\'auto\')\n    a = p.parse_args(); seed_all()\n    if a.epochs < 1 or a.batch_size < 2 or a.size < 32 or a.workers < 0 or a.lr <= 0:\n        p.error(\'epochs/lr must be positive, batch-size >= 2, size >= 32, workers >= 0\')\n    if a.cache_features and (a.fine_tune or a.from_scratch):\n        p.error(\'--cache-features requires pretrained classifier-head training\')\n    if a.selection_source != \'combined\' and a.cache_features:\n        p.error(\'Source-specific selection is only supported for image-based training\')\n    if a.from_scratch and not a.fine_tune:\n        p.error(\'--from-scratch requires --fine-tune so random features are not frozen\')\n    if a.device == \'cuda\' and not torch.cuda.is_available():\n        raise RuntimeError(\'CUDA requested, but no CUDA GPU is available\')\n    device = torch.device(\'cuda\' if a.device == \'auto\' and torch.cuda.is_available() else\n                          (\'cpu\' if a.device == \'auto\' else a.device))\n    out = Path(a.output_dir); checkpoint = out / \'skinsight_mobilenetv3_small.pt\'\n    if checkpoint.exists() or (out / \'test_metrics.json\').exists():\n        raise FileExistsError(\'Training output already exists; use a new --output-dir to preserve prior results\')\n    df = pd.read_csv(a.manifest)\n    required = {\'image_path\', \'label\', \'source\', \'original_id\'}\n    if required - set(df.columns):\n        raise ValueError(f\'Manifest lacks columns: {sorted(required-set(df.columns))}\')\n    if not set(df.label).issubset(CLASS_NAMES):\n        raise ValueError(\'Manifest contains unsupported labels\')\n    if \'image_sha256\' not in df or df.image_sha256.isna().any():\n        raise ValueError(\'Rebuild the manifest to include image_sha256 for duplicate-aware splitting\')\n    tr, va, test = split_three_way(df)\n    out.mkdir(parents=True, exist_ok=True)\n    for name, part in [(\'train\', tr), (\'validation\', va), (\'test\', test)]:\n        part.to_csv(out / f\'{name}_split.csv\', index=False)\n        print(name, len(part)); print(part.groupby([\'source\', \'label\']).size().unstack(fill_value=0))\n    grouping = df.groupby([\'source\', \'grouping_level\']).size().to_dict() if \'grouping_level\' in df else {}\n    split_info = {\n        \'seed\': 42, \'manifest_sha256\': hashlib.sha256(Path(a.manifest).read_bytes()).hexdigest(),\n        \'counts\': {\'train\': len(tr), \'validation\': len(va), \'test\': len(test)},\n        \'grouping_coverage\': {f\'{source}::{level}\': int(count) for (source, level), count in grouping.items()},\n        \'limitations\': \'Linked patient, lesion and exact-file hashes stay together. Image/lesion groups do not prove patient separation. Re-encoded near-duplicates and unseen external data need further evaluation.\',\n    }\n    (out / \'split_summary.json\').write_text(json.dumps(split_info, indent=2))\n    fitting_frame = tr if a.train_source == \'combined\' else tr[tr.source == a.train_source].copy()\n    if fitting_frame.empty or set(fitting_frame.label) != set(CLASS_NAMES):\n        raise ValueError(\'Fitting source must have training samples for all six classes\')\n    split_info[\'fitting_source\'] = a.train_source\n    split_info[\'fitting_records\'] = len(fitting_frame)\n    (out / \'split_summary.json\').write_text(json.dumps(split_info, indent=2))\n    print(\'Optimization source:\', a.train_source, \'training records:\', len(fitting_frame), flush=True)\n    model = create_model(pretrained=not a.from_scratch and not bool(a.init_checkpoint), freeze_backbone=not a.fine_tune).to(device)\n    if a.init_checkpoint:\n        initial = torch.load(a.init_checkpoint, map_location=device, weights_only=True)\n        if initial.get(\'architecture\') != \'mobilenet_v3_small\' or initial.get(\'class_names\') != CLASS_NAMES:\n            raise ValueError(\'Warm-start checkpoint architecture/classes do not match\')\n        if int(initial.get(\'image_size\', a.size)) != a.size:\n            raise ValueError(\'Warm-start image size differs from this run\')\n        model.load_state_dict(initial[\'model_state\'])\n    selection_frame = va if a.selection_source == \'combined\' else va[va.source == a.selection_source]\n    if selection_frame.empty:\n        raise ValueError(\'Selection source has no validation samples\')\n    training_dataset = ManifestDataset(fitting_frame, tfms(True, a.size))\n    validation_dataset = ManifestDataset(va, tfms(False, a.size))\n    training_network = model\n    if a.cache_features:\n        print(\'Caching frozen features; training augmentation disabled for this baseline\', flush=True)\n        training_dataset = cache_features(model, fitting_frame, tfms(False, a.size), device, a.batch_size, a.workers)\n        validation_dataset = cache_features(model, va, tfms(False, a.size), device, a.batch_size, a.workers)\n        training_network = model.classifier\n    training = DataLoader(training_dataset, batch_size=a.batch_size,\n                          sampler=sampler(fitting_frame), num_workers=0 if a.cache_features else a.workers,\n                          pin_memory=device.type == \'cuda\', drop_last=len(fitting_frame) % a.batch_size == 1)\n    validation = DataLoader(validation_dataset, batch_size=a.batch_size,\n                            shuffle=False, num_workers=0 if a.cache_features else a.workers)\n    selection_loader = DataLoader(ManifestDataset(selection_frame, tfms(False, a.size)),\n                                  batch_size=a.batch_size, shuffle=False, num_workers=a.workers)\n    optimizer = torch.optim.AdamW([x for x in model.parameters() if x.requires_grad], lr=a.lr, weight_decay=1e-4)\n    criterion = nn.CrossEntropyLoss(label_smoothing=.05)\n    best = -1.; history = []; best_epoch = None\n    if a.init_checkpoint:\n        initial_loss, best, initial_report = evaluate(model, selection_loader if a.selection_source != \'combined\' else validation, device)\n        best_epoch = 0\n        initial.update(balanced_accuracy=best, validation_report=initial_report, selection_source=a.selection_source)\n        torch.save(initial, checkpoint)\n        history.append({\'epoch\': 0, \'validation_loss\': initial_loss, \'validation_balanced_accuracy\': best})\n        print(\'Warm-start validation reference:\', best, flush=True)\n    print(\'Device:\', device, \'Architecture: mobilenet_v3_small\', flush=True)\n    for epoch in range(1, a.epochs + 1):\n        start = time.time(); model.train(); run = 0.; seen = 0\n        if not a.fine_tune:\n            model.features.eval()\n        for i, (x, y) in enumerate(training, 1):\n            x, y = x.to(device), y.to(device); optimizer.zero_grad(set_to_none=True)\n            logits = training_network(x); loss = criterion(logits, y); loss.backward(); optimizer.step()\n            run += float(loss.detach()) * len(y); seen += len(y)\n            if i % 100 == 0 or i == len(training):\n                print(f\'epoch {epoch}/{a.epochs}, batch {i}/{len(training)}\', flush=True)\n        vl, score, report = evaluate(training_network, validation if a.selection_source == "combined" else selection_loader, device)\n        history.append({\'epoch\': epoch, \'train_loss\': run / seen, \'validation_loss\': vl,\n                        \'validation_balanced_accuracy\': score, \'seconds\': time.time() - start})\n        print(history[-1], flush=True)\n        if score > best:\n            best = score; best_epoch = epoch\n            torch.save({\'model_state\': model.state_dict(), \'class_names\': CLASS_NAMES,\n                        \'architecture\': \'mobilenet_v3_small\', \'image_size\': a.size,\n                        \'balanced_accuracy\': best, \'validation_report\': report,\n                        \'selection_source\': a.selection_source, \'fitting_source\': a.train_source,\n                        \'init_checkpoint\': Path(a.init_checkpoint).name if a.init_checkpoint else None,\n                        \'dataset\': \'PAD-UFES-20 + ISIC-2019\',\n                        \'training_mode\': \'imagenet_cached_head\' if a.cache_features else (\'from_scratch\' if a.from_scratch else (\'imagenet_finetune\' if a.fine_tune else \'imagenet_head_only\'))}, checkpoint)\n    # Only after selection, reload the best checkpoint and evaluate test data once.\n    best_checkpoint = torch.load(checkpoint, map_location=device, weights_only=True)\n    model.load_state_dict(best_checkpoint[\'model_state\'])\n    validation_report = report_frame(training_network, va, a.size, a.batch_size, a.workers, device,\n                                     validation_dataset if a.cache_features else None)\n    if not a.skip_test:\n        test_report = report_frame(model, test, a.size, a.batch_size, a.workers, device)\n        test_report[\'evaluation_note\'] = \'Held out from fitting and checkpoint selection. Reusing this test set to tune the model invalidates independent-test claims. Patient independence depends on grouping coverage.\'\n        test_report[\'split_summary\'] = split_info\n        (out / \'test_metrics.json\').write_text(json.dumps(test_report, indent=2))\n    (out / \'metrics.json\').write_text(json.dumps({\'best_epoch\': best_epoch, \'best_balanced_accuracy\': best,\n                                                \'history\': history, \'validation\': validation_report,\n                                                \'selection_source\': a.selection_source, \'fitting_source\': a.train_source,\n                                                \'fitting_records\': len(fitting_frame), \'test_evaluated\': not a.skip_test}, indent=2))\n    print(\'Saved:\', checkpoint, \'test evaluated:\', not a.skip_test, flush=True)\n\n\nif __name__ == \'__main__\':\n    main()\n', 'ml/external_data.py': '"""Audit a new evaluation/calibration manifest against all previously used records."""\nimport hashlib\nfrom pathlib import Path\nimport pandas as pd\nfrom dataset import CLASS_NAMES\nfrom splits import linked_groups\n\n\ndef audit_external(manifest, reference_dir, additional_reference=None):\n    frame = pd.read_csv(manifest).reset_index(drop=True)\n    required = {\'image_path\', \'label\', \'source\', \'original_id\', \'image_sha256\'}\n    if required - set(frame.columns) or frame.empty:\n        raise ValueError(\'External manifest must be nonempty and include \' + \', \'.join(sorted(required)))\n    if frame[list(required)].isna().any().any() or not set(frame.label).issubset(CLASS_NAMES):\n        raise ValueError(\'External manifest contains missing fields or unsupported labels\')\n    if frame.duplicated([\'source\', \'original_id\']).any():\n        raise ValueError(\'External manifest contains duplicate source/image identities\')\n    for row in frame.itertuples():\n        with Path(row.image_path).open(\'rb\') as file:\n            actual = hashlib.file_digest(file, \'sha256\').hexdigest()\n        if actual != row.image_sha256:\n            raise ValueError(f\'Image checksum mismatch for {row.original_id}\')\n    references = []\n    for name in [\'train\', \'validation\', \'test\']:\n        path = Path(reference_dir) / f\'{name}_split.csv\'\n        if not path.is_file():\n            raise FileNotFoundError(f\'All three reference partitions are required: {path}\')\n        references.append(pd.read_csv(path))\n    if additional_reference is not None:\n        references.append(pd.read_csv(additional_reference))\n    known = pd.concat(references, ignore_index=True)\n    combined = pd.concat([known, frame], ignore_index=True)\n    groups = linked_groups(combined)\n    if set(groups.iloc[:len(known)]) & set(groups.iloc[len(known):]):\n        raise ValueError(\'External data overlaps previously used patient/lesion/image/hash groups\')\n    coverage = frame.groupby([\'source\', \'label\']).size()\n    audit = {\'records\': len(frame), \'known_group_overlap\': False,\n             \'source_class_counts\': {f\'{source}::{label}\': int(n) for (source, label), n in coverage.items()},\n             \'manifest_sha256\': hashlib.sha256(Path(manifest).read_bytes()).hexdigest(),\n             \'limitations\': \'No overlap of recorded identities or exact-file hashes. Missing/aliased patient IDs and re-encoded near-duplicates can still hide overlap; source independence and ground-truth provenance require review.\'}\n    return frame, audit\n', 'ml/calibrate.py': '"""Temperature scaling on NEW calibration data, separate from reported evaluation."""\nimport argparse\nimport json\nimport sys\nfrom pathlib import Path\nimport torch\nfrom torch.utils.data import DataLoader\nsys.path.insert(0, str(Path(__file__).resolve().parents[1]))\nfrom app import inference\nfrom dataset import ManifestDataset\nfrom train import tfms\nfrom external_data import audit_external\n\n\ndef fit_temperature(logits, labels):\n    """One positive scalar; preserves class ordering and cannot increase accuracy."""\n    logits = logits.detach().cpu().double(); labels = labels.detach().cpu()\n    log_temperature = torch.zeros((), dtype=torch.float64, requires_grad=True)\n    optimizer = torch.optim.LBFGS([log_temperature], lr=.1, max_iter=100, line_search_fn=\'strong_wolfe\')\n    def closure():\n        optimizer.zero_grad()\n        loss = torch.nn.functional.cross_entropy(logits / log_temperature.clamp(-3, 3).exp(), labels)\n        loss.backward(); return loss\n    before = float(torch.nn.functional.cross_entropy(logits, labels))\n    optimizer.step(closure)\n    temperature = float(log_temperature.detach().clamp(-3, 3).exp())\n    after = float(torch.nn.functional.cross_entropy(logits / temperature, labels))\n    if not torch.isfinite(torch.tensor(after)) or after > before + 1e-8:\n        return 1., before, before\n    return temperature, before, after\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--manifest\', required=True, help=\'New calibration cohort, not an already reported test\')\n    parser.add_argument(\'--checkpoint\', required=True)\n    parser.add_argument(\'--reference-dir\', required=True, help=\'Run with train/validation/test split CSVs\')\n    parser.add_argument(\'--output-checkpoint\', required=True)\n    args = parser.parse_args()\n    output = Path(args.output_checkpoint)\n    if output.exists():\n        raise FileExistsError(\'Choose a new calibration output; existing checkpoints are preserved\')\n    frame, audit = audit_external(args.manifest, args.reference_dir)\n    if \'patient_id\' not in frame or frame.patient_id.isna().any():\n        raise ValueError(\'Calibration requires real patient groups to separate calibration from subsequent testing\')\n    patients = frame.patient_id.astype(str).str.strip()\n    if patients.eq(\'\').any() or patients.str.lower().str.startswith(\'dummy\').any():\n        raise ValueError(\'Calibration requires genuine patient IDs, not empty or dummy IDs\')\n    inference.CHECKPOINT = Path(args.checkpoint)\n    predictor = inference.Predictor()\n    if not predictor.ready:\n        raise FileNotFoundError(\'Checkpoint missing\')\n    if predictor.meta.get(\'calibration\'):\n        raise ValueError(\'Start from an uncalibrated checkpoint; repeated calibration would obscure cohort provenance\')\n    logits, labels = [], []\n    loader = DataLoader(ManifestDataset(frame, tfms(False, predictor.image_size)), batch_size=32)\n    with torch.no_grad():\n        for images, targets in loader:\n            logits.append(predictor.model(images.to(predictor.device)).cpu())\n            labels.append(targets)\n    temperature, before, after = fit_temperature(torch.cat(logits), torch.cat(labels))\n    checkpoint = torch.load(args.checkpoint, map_location=\'cpu\', weights_only=True)\n    checkpoint[\'temperature\'] = temperature\n    checkpoint[\'calibration\'] = {\'method\': \'temperature_scaling\', \'audit\': audit,\n                                 \'negative_log_likelihood_before\': before,\n                                 \'negative_log_likelihood_after\': after,\n                                 \'clinical_validation\': False,\n                                 \'note\': \'Fit on a calibration cohort; improvement on this same cohort does not prove external calibration. Never evaluate using the calibration data itself.\'}\n    output.parent.mkdir(parents=True, exist_ok=True)\n    torch.save(checkpoint, output)\n    # Preserve cohort identity records so a later external test can also exclude them.\n    frame.to_csv(output.with_suffix(\'.calibration.csv\'), index=False)\n    print(json.dumps({\'temperature\': temperature, \'calibration_nll_before\': before, \'calibration_nll_after\': after}))\n\n\nif __name__ == \'__main__\': main()\n', 'ml/evaluate_external.py': '"""Evaluate new external data; never treat the old reported test as a new test."""\nimport argparse\nimport json\nimport sys\nfrom pathlib import Path\nimport torch\nsys.path.insert(0, str(Path(__file__).resolve().parents[1]))\nfrom app import inference\nfrom train import report_frame\nfrom external_data import audit_external\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--manifest\', required=True)\n    parser.add_argument(\'--checkpoint\', required=True)\n    parser.add_argument(\'--reference-dir\', required=True)\n    parser.add_argument(\'--output\', required=True)\n    args = parser.parse_args()\n    output = Path(args.output)\n    if output.exists():\n        raise FileExistsError(\'Preserve previous evaluation reports; choose a new output\')\n    inference.CHECKPOINT = Path(args.checkpoint)\n    predictor = inference.Predictor()\n    if not predictor.ready:\n        raise FileNotFoundError(\'Checkpoint missing\')\n    calibration_reference = None\n    if predictor.meta.get("calibration"):\n        calibration_reference = Path(args.checkpoint).with_suffix(".calibration.csv")\n        if not calibration_reference.exists():\n            raise FileNotFoundError("Calibrated model requires its calibration identity CSV to exclude that cohort from testing")\n    frame, audit = audit_external(args.manifest, args.reference_dir, calibration_reference)\n    # report_frame applies temperature to logits when the checkpoint supplies it.\n    model = predictor.model\n    if predictor.temperature != 1:\n        class ScaledModel(torch.nn.Module):\n            def __init__(self, model, temperature):\n                super().__init__(); self.model = model; self.temperature = temperature\n            def forward(self, x):\n                return self.model(x) / self.temperature\n        model = ScaledModel(model, predictor.temperature)\n    report = report_frame(model, frame, predictor.image_size, 32, 0, predictor.device)\n    report[\'external_audit\'] = audit\n    report[\'evaluation_note\'] = \'External research evaluation. Identity checks alone do not certify patient independence or clinical validity.\'\n    output.parent.mkdir(parents=True, exist_ok=True)\n    output.write_text(json.dumps(report, indent=2))\n    print(\'External balanced accuracy:\', report[\'balanced_accuracy\'])\n\n\nif __name__ == \'__main__\': main()\n', 'ml/features.py': '"""Frozen ImageNet feature caching for a fast CPU classifier-head baseline."""\nimport torch\nfrom torch.utils.data import DataLoader, TensorDataset\nfrom dataset import ManifestDataset\n\n\n@torch.no_grad()\ndef cache_features(model, frame, transform, device, batch_size, workers):\n    model.eval()\n    loader = DataLoader(ManifestDataset(frame, transform), batch_size=batch_size,\n                        shuffle=False, num_workers=workers)\n    features, labels = [], []\n    for i, (images, targets) in enumerate(loader, 1):\n        values = model.avgpool(model.features(images.to(device))).flatten(1)\n        features.append(values.cpu()); labels.append(targets)\n        if i % 100 == 0 or i == len(loader):\n            print(f\'Frozen feature extraction {i}/{len(loader)}\', flush=True)\n    return TensorDataset(torch.cat(features), torch.cat(labels))\n', 'tests/test_api.py': '"""Run the actual HTTP API with a temporary artificial checkpoint."""\nimport io\nimport http.cookiejar\nimport re\nimport urllib.parse\nimport json\nimport os\nimport socket\nimport sys\nimport tempfile\nimport threading\nimport time\nimport unittest\nimport urllib.error\nimport urllib.request\nfrom pathlib import Path\nfrom unittest.mock import patch\nsys.path.insert(0, str(Path(__file__).resolve().parents[1]))\nsys.path.insert(0, str(Path(__file__).resolve().parents[1] / \'ml\'))\nos.environ[\'NO_ALBUMENTATIONS_UPDATE\'] = \'1\'\nimport numpy as np\nimport torch\nimport uvicorn\nfrom PIL import Image\nfrom model import create_model, CLASS_NAMES\nfrom app import inference, main, auth\n\nclass APITests(unittest.TestCase):\n    @classmethod\n    def setUpClass(cls):\n        torch.set_num_threads(2)\n        cls.temp = tempfile.TemporaryDirectory()\n        cls.auth_patch=patch.object(auth,\'DB_PATH\',Path(cls.temp.name)/\'accounts.sqlite3\');cls.auth_patch.start()\n        checkpoint=Path(cls.temp.name)/\'test.pt\'\n        torch.save(dict(model_state=create_model(pretrained=False).state_dict(),\n                        class_names=CLASS_NAMES,architecture=\'mobilenet_v3_small\',image_size=64),checkpoint)\n        with patch.object(inference,\'CHECKPOINT\',checkpoint):\n            predictor=inference.Predictor()\n        cls.predictor_patch=patch.object(main,\'predictor\',predictor);cls.predictor_patch.start()\n        cls.sock=socket.socket();cls.sock.bind((\'127.0.0.1\',0))\n        cls.url=f\'http://127.0.0.1:{cls.sock.getsockname()[1]}\'\n        cls.server=uvicorn.Server(uvicorn.Config(main.app,log_level=\'error\'))\n        cls.thread=threading.Thread(target=cls.server.run,kwargs={\'sockets\':[cls.sock]},daemon=True)\n        cls.thread.start()\n        for _ in range(100):\n            if cls.server.started: break\n            time.sleep(.02)\n        if not cls.server.started: raise RuntimeError(\'API did not start\')\n        cls.opener=urllib.request.build_opener(urllib.request.HTTPCookieProcessor(http.cookiejar.CookieJar()))\n        html=cls.opener.open(cls.url+\'/register\').read().decode()\n        cls.csrf=re.search(r\'name="csrf" value="([^"]+)"\',html).group(1)\n        form=urllib.parse.urlencode(dict(username=\'testuser\',password=\'unique-test-password\',email=\'testuser@example.com\',csrf=cls.csrf)).encode()\n        cls.opener.open(cls.url+\'/register\',form).close()\n        cls.opener.open(cls.url+\'/login\',form).close()\n\n    @classmethod\n    def tearDownClass(cls):\n        cls.server.should_exit=True;cls.thread.join(timeout=5)\n        cls.sock.close();cls.predictor_patch.stop();cls.auth_patch.stop();cls.temp.cleanup()\n\n    def upload(self,raw,content_type=\'image/png\'):\n        boundary=\'skinsighttest\'\n        body=(f\'--{boundary}\\r\\nContent-Disposition: form-data; name="image"; filename="image.png"\\r\\nContent-Type: {content_type}\\r\\n\\r\\n\').encode()+raw+f\'\\r\\n--{boundary}--\\r\\n\'.encode()\n        return self.opener.open(urllib.request.Request(self.url+\'/api/predict\',data=body,headers={\'Content-Type\':f\'multipart/form-data; boundary={boundary}\', \'X-CSRF-Token\':self.csrf}))\n\n    def test_prediction_and_html(self):\n        with self.opener.open(self.url+\'/\') as response:\n            self.assertIn(b\'MobileNet\',response.read())\n        with urllib.request.urlopen(self.url+\'/api/health\') as response:\n            self.assertTrue(json.load(response)[\'model_ready\'])\n        raw=io.BytesIO();Image.fromarray(np.random.default_rng(42).integers(50,200,(128,128,3),dtype=np.uint8)).save(raw,format=\'PNG\')\n        with self.upload(raw.getvalue()) as response:\n            result=json.load(response)\n            self.assertIn(result[\'label\'],CLASS_NAMES)\n            self.assertIn(\'guidance\',result)\n            self.assertIn(\'stage\',result)\n            self.assertIn(\'not a medical diagnosis\',result[\'medical_disclaimer\'])\n        with self.opener.open(self.url+\'/reports/\'+result[\'report_id\']) as response:\n            self.assertIn(b\'SAVED SCREENING\',response.read())\n        with self.opener.open(self.url+result[\'report_url\']) as response:\n            self.assertEqual(response.headers[\'Content-Type\'],\'application/pdf\')\n            self.assertTrue(response.read().startswith(b\'%PDF-1.4\'))\n        with self.assertRaises(urllib.error.HTTPError) as error:\n            urllib.request.urlopen(self.url+result[\'report_url\'])\n        self.assertEqual(error.exception.code,401)\n        with self.opener.open(self.url+result[\'email_status_url\']) as response:\n            self.assertEqual(json.load(response)[\'email_status\'],\'not_configured\')\n\n    def test_public_pages_and_dashboard(self):\n        for path in ["/home","/about","/faq","/stages"]:\n            with urllib.request.urlopen(self.url+path) as response:\n                self.assertEqual(response.status,200)\n                self.assertIn(b"SkinSight",response.read())\n        with self.opener.open(self.url+"/dashboard") as response:\n            self.assertIn(b"testuser",response.read())\n        with urllib.request.urlopen(self.url+"/dashboard") as response:\n            self.assertTrue(response.url.endswith("/login"))\n\n    def test_avatar_rejected_before_inference(self):\n        from PIL import ImageDraw\n        avatar=Image.new("RGB",(128,128),(128,140,140))\n        ImageDraw.Draw(avatar).text((35,45),"SP",fill="white")\n        raw=io.BytesIO();avatar.save(raw,format="PNG")\n        with patch.object(main.predictor,"predict_bytes") as prediction:\n            with self.assertRaises(urllib.error.HTTPError) as error: self.upload(raw.getvalue())\n            self.assertEqual(error.exception.code,422)\n            self.assertIn("graphic",json.load(error.exception)["detail"])\n            prediction.assert_not_called()\n\n    def test_unusable_photos_rejected(self):\n        for image in [Image.new("RGB",(32,32),"brown"), Image.new("RGB",(128,128),"black"), Image.new("RGB",(128,128),"white")]:\n            with self.subTest(size=image.size):\n                raw=io.BytesIO();image.save(raw,format="PNG")\n                with patch.object(main.predictor,"predict_bytes") as prediction:\n                    with self.assertRaises(urllib.error.HTTPError) as error: self.upload(raw.getvalue())\n                    self.assertEqual(error.exception.code,422)\n                    prediction.assert_not_called()\n\n    def test_authentication_required(self):\n        with self.assertRaises(urllib.error.HTTPError) as error:\n            urllib.request.urlopen(urllib.request.Request(self.url+"/api/predict",data=b"image=invalid"))\n        self.assertEqual(error.exception.code,401)\n\n    def test_invalid_image_rejected(self):\n        with self.assertRaises(urllib.error.HTTPError) as error: self.upload(b\'invalid\')\n        self.assertEqual(error.exception.code,400)\n\n    def test_untrained_model_returns_503(self):\n        with patch.object(main.predictor,\'model\',None):\n            with self.assertRaises(urllib.error.HTTPError) as error: self.upload(b\'invalid\')\n        self.assertEqual(error.exception.code,503)\n\nif __name__ == \'__main__\': unittest.main()\n', 'tests/test_evaluation.py': "import sys\nimport unittest\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nsys.path.insert(0, str(Path(__file__).resolve().parents[1] / 'ml'))\nfrom splits import linked_groups, split_three_way\nfrom evaluation import summarize\n\nclass EvaluationTests(unittest.TestCase):\n    def test_transitive_patient_lesion_and_duplicate_grouping(self):\n        frame = pd.DataFrame([\n            dict(source='PAD', original_id='a',patient_id='p1',lesion_id='l1',image_sha256='hashA'),\n            dict(source='PAD', original_id='b',patient_id='p1',lesion_id='l2',image_sha256='hashB'),\n            dict(source='ISIC',original_id='c',patient_id='',lesion_id='l3',image_sha256='hashB'),\n            dict(source='ISIC',original_id='d',patient_id='',lesion_id='l3',image_sha256='hashD'),\n            dict(source='ISIC',original_id='e',patient_id='',lesion_id='',image_sha256='unique'),\n        ])\n        groups = linked_groups(frame)\n        self.assertEqual(len(set(groups[:4])),1)\n        self.assertNotEqual(groups.iloc[0],groups.iloc[4])\n\n    def test_three_partitions_are_disjoint_and_keep_all_strata(self):\n        frame=pd.DataFrame([dict(source=source,label=label,original_id=f'{source}_{label}_{i}',\n                                 patient_id=f'{source}_{label}_{i//2}',image_sha256=f'{source}_{label}_{i}')\n                            for source in ['PAD','ISIC'] for label in ['MEL','NEV'] for i in range(30)])\n        parts=split_three_way(frame)\n        for i,part in enumerate(parts):\n            self.assertEqual(set(part.label),{'MEL','NEV'})\n            self.assertEqual(set(part.source),{'PAD','ISIC'})\n            for other in parts[:i]:\n                self.assertFalse(set(part._group)&set(other._group))\n        self.assertEqual(sum(map(len,parts)),len(frame))\n        again=split_three_way(frame)\n        self.assertEqual(parts[2].original_id.tolist(),again[2].original_id.tolist())\n\n    def test_specificity_sensitivity_auc_and_confusion_matrix(self):\n        probabilities=np.array([[.8,.2,0,0,0,0],[.1,.9,0,0,0,0],[.7,.3,0,0,0,0]])\n        report=summarize([0,1,1],probabilities)\n        self.assertEqual(report['confusion_matrix'][1][0],1)\n        self.assertEqual(report['per_class']['BCC']['sensitivity'],.5)\n        self.assertEqual(report['per_class']['ACK']['specificity'],.5)\n        self.assertIsNone(report['per_class']['MEL']['sensitivity'])\n        self.assertIsNone(report['per_class']['MEL']['roc_auc'])\n        self.assertGreaterEqual(report['expected_calibration_error_10_bins'],0)\n\nif __name__ == '__main__': unittest.main()\n", 'tests/test_workflow.py': '"""Pipeline regression tests use artificial images, never medical accuracy claims."""\nimport io\nimport os\nimport sys\nimport tempfile\nimport unittest\nfrom pathlib import Path\nfrom unittest.mock import patch\n\nROOT = Path(__file__).resolve().parents[1]\nsys.path.insert(0, str(ROOT))\nsys.path.insert(0, str(ROOT / \'ml\'))\nos.environ[\'NO_ALBUMENTATIONS_UPDATE\'] = \'1\'\nimport pandas as pd\nimport torch\nfrom PIL import Image\nfrom model import create_model\nfrom train import split_df, evaluate, tfms\nfrom dataset import ManifestDataset, CLASS_NAMES\nfrom prepare_combined import build_pad_manifest, build_isic_manifest\nfrom torch.utils.data import DataLoader\nfrom app import inference\n\ntorch.set_num_threads(2)\n\nclass WorkflowTests(unittest.TestCase):\n    def test_training_checkpoint_loads_and_predicts(self):\n        with tempfile.TemporaryDirectory() as tmp:\n            checkpoint = Path(tmp) / \'model.pt\'\n            model = create_model(pretrained=False, freeze_backbone=False)\n            torch.save(dict(model_state=model.state_dict(), class_names=CLASS_NAMES,\n                            architecture=\'mobilenet_v3_small\', image_size=64,\n                            validation_report={\'value\': 0.2}), checkpoint)\n            with patch.object(inference, \'CHECKPOINT\', checkpoint):\n                predictor = inference.Predictor()\n            raw = io.BytesIO()\n            Image.new(\'RGB\', (64, 64), \'brown\').save(raw, format=\'PNG\')\n            result = predictor.predict_bytes(raw.getvalue())\n            self.assertTrue(predictor.ready)\n            self.assertIn(result[\'label\'], CLASS_NAMES)\n            self.assertEqual(len(result[\'top_predictions\']), 3)\n            self.assertTrue(0 <= result[\'confidence\'] <= 100)\n\n    def test_patient_split_no_overlap(self):\n        rows = [dict(label=CLASS_NAMES[i % 6], source=\'PAD-UFES-20\',\n                     patient_id=f\'PAD::{i // 2}\', original_id=str(i)) for i in range(60)]\n        train, val = split_df(pd.DataFrame(rows))\n        self.assertFalse(set(train.patient_id) & set(val.patient_id))\n        self.assertEqual(len(train) + len(val), 60)\n\n    def test_recursive_manifest_mapping_and_missing_image(self):\n        with tempfile.TemporaryDirectory() as tmp:\n            root = Path(tmp); nested = root / \'imgs_part_1\'; nested.mkdir()\n            Image.new(\'RGB\', (64,64)).save(nested / \'PAD_1.png\')\n            Image.new(\'RGB\', (64,64)).save(root / \'ISIC_1.jpg\')\n            pd.DataFrame([dict(img_id=\'PAD_1.png\', diagnostic=\'NEV\', patient_id=7)]).to_csv(root/\'pad.csv\',index=False)\n            pd.DataFrame([dict(image=\'ISIC_1\', NV=1, MEL=0, DF=0),dict(image=\'excluded\', NV=0, MEL=0, DF=1)]).to_csv(root/\'isic.csv\',index=False)\n            pad=build_pad_manifest(root/\'pad.csv\',root)\n            isic=build_isic_manifest(root/\'isic.csv\',root)\n            self.assertEqual(pad.iloc[0].patient_id, \'PAD::7\')\n            self.assertEqual(isic.iloc[0].label, \'NEV\')\n            self.assertEqual(len(isic),1)\n            (nested/\'PAD_1.png\').unlink()\n            with self.assertRaises(ValueError): build_pad_manifest(root/\'pad.csv\',root)\n\n    def test_isic_metadata_uses_lesions_and_rejects_dummy_patients(self):\n        with tempfile.TemporaryDirectory() as tmp:\n            root=Path(tmp)\n            Image.new(\'RGB\',(64,64)).save(root/\'ISIC_1.jpg\')\n            pd.DataFrame([dict(image=\'ISIC_1\', NV=1)]).to_csv(root/\'gt.csv\',index=False)\n            pd.DataFrame([dict(isic_id=\'ISIC_1\',patient_id=\'dummy_0\',lesion_id=\'lesion_7\')]).to_csv(root/\'meta.csv\',index=False)\n            manifest=build_isic_manifest(root/\'gt.csv\',root,root/\'meta.csv\')\n            self.assertEqual(manifest.iloc[0].patient_id,\'\')\n            self.assertEqual(manifest.iloc[0].lesion_id,\'lesion_7\')\n            self.assertEqual(manifest.iloc[0].grouping_level,\'lesion\')\n            self.assertEqual(len(manifest.iloc[0].image_sha256),64)\n\n    def test_conflicting_isic_labels_are_rejected(self):\n        with tempfile.TemporaryDirectory() as tmp:\n            root=Path(tmp)\n            pd.DataFrame([dict(image=\'ISIC_1\', NV=1, DF=1)]).to_csv(root/\'gt.csv\',index=False)\n            with self.assertRaisesRegex(ValueError,\'one-hot\'):\n                build_isic_manifest(root/\'gt.csv\',root)\n\n    def test_cached_features_match_full_model_logits(self):\n        from features import cache_features\n        with tempfile.TemporaryDirectory() as tmp:\n            path=Path(tmp)/\'image.png\'; Image.new(\'RGB\',(64,64),\'brown\').save(path)\n            frame=pd.DataFrame([dict(image_path=str(path),label=\'NEV\')])\n            model=create_model(pretrained=False).eval()\n            cached=cache_features(model,frame,tfms(False,64),torch.device(\'cpu\'),2,0)\n            image,_=ManifestDataset(frame,tfms(False,64))[0]\n            with torch.no_grad():\n                full=model(image.unsqueeze(0))\n                head=model.classifier(cached.tensors[0])\n            torch.testing.assert_close(full,head)\n            self.assertEqual(cached.tensors[1].tolist(),[3])\n\n    def test_evaluation_handles_absent_classes(self):\n        with tempfile.TemporaryDirectory() as tmp:\n            image=Path(tmp)/\'one.png\'; Image.new(\'RGB\',(64,64)).save(image)\n            ds=ManifestDataset(pd.DataFrame([dict(image_path=str(image),label=\'NEV\')]),tfms(False,64))\n            loss,score,report=evaluate(create_model(pretrained=False),DataLoader(ds,batch_size=1),torch.device(\'cpu\'))\n            self.assertIn(\'MEL\',report)\n            self.assertTrue(loss >= 0)\n\nif __name__ == \'__main__\': unittest.main()\n', 'tests/test_finetune.py': '"""A CLI run proves warm-start training never decodes old test images in skip mode."""\nimport hashlib\nimport os\nimport subprocess\nimport sys\nimport tempfile\nimport unittest\nfrom pathlib import Path\nimport pandas as pd\nimport torch\nfrom PIL import Image\nsys.path.insert(0,str(Path(__file__).resolve().parents[1]/\'ml\'))\nfrom model import create_model,CLASS_NAMES\nfrom splits import split_three_way\n\nclass FineTuneTests(unittest.TestCase):\n    def test_warm_start_and_skip_test(self):\n        with tempfile.TemporaryDirectory() as temp:\n            root=Path(temp);rows=[]\n            for label_index,label in enumerate(CLASS_NAMES):\n                for i in range(15):\n                    image=root/f\'{label}_{i}.png\'\n                    Image.new(\'RGB\',(64,64),(label_index*40,i*12,90)).save(image)\n                    rows.append(dict(image_path=str(image),label=label,source=\'PAD-UFES-20\',\n                                     patient_id=f\'{label}_{i}\',original_id=f\'{label}_{i}\',\n                                     image_sha256=hashlib.sha256(image.read_bytes()).hexdigest()))\n            frame=pd.DataFrame(rows);_,_,test=split_three_way(frame)\n            # Keep metadata and hashes but remove test files; decoding them would fail.\n            for path in test.image_path:Path(path).unlink()\n            frame.to_csv(root/\'manifest.csv\',index=False)\n            checkpoint=root/\'initial.pt\'\n            torch.save(dict(model_state=create_model(pretrained=False).state_dict(),\n                            architecture=\'mobilenet_v3_small\',class_names=CLASS_NAMES,\n                            image_size=64,training_mode=\'synthetic\'),checkpoint)\n            env=os.environ.copy();env.update(OMP_NUM_THREADS=\'1\',MKL_NUM_THREADS=\'1\',NO_ALBUMENTATIONS_UPDATE=\'1\')\n            result=subprocess.run([sys.executable,str(Path(__file__).resolve().parents[1]/\'ml/train.py\'),\n                \'--manifest\',str(root/\'manifest.csv\'),\'--init-checkpoint\',str(checkpoint),\n                \'--fine-tune\',\'--skip-test\',\'--train-source\',\'PAD-UFES-20\',\'--selection-source\',\'PAD-UFES-20\',\n                \'--epochs\',\'1\',\'--size\',\'64\',\'--batch-size\',\'8\',\'--workers\',\'0\',\n                \'--device\',\'cpu\',\'--output-dir\',str(root/\'output\')],env=env,capture_output=True,text=True,timeout=90)\n            self.assertEqual(result.returncode,0,result.stdout+result.stderr)\n            self.assertTrue((root/\'output/metrics.json\').exists())\n            self.assertFalse((root/\'output/test_metrics.json\').exists())\n            import json\n            metrics=json.loads((root/\'output/metrics.json\').read_text())\n            self.assertFalse(metrics[\'test_evaluated\'])\n            self.assertEqual(metrics[\'fitting_source\'],\'PAD-UFES-20\')\n            self.assertEqual(metrics[\'fitting_records\'],len(pd.read_csv(root/\'output/train_split.csv\')))\n            self.assertEqual(metrics[\'history\'][0][\'epoch\'],0)\n\nif __name__ == \'__main__\':unittest.main()\n', 'tests/test_dashboard.py': "import sys\nimport tempfile\nimport unittest\nfrom pathlib import Path\nfrom unittest.mock import patch\nsys.path.insert(0,str(Path(__file__).resolve().parents[1]))\nfrom app import auth,reporting\nfrom app.guidance import GUIDANCE,stage_info\n\nclass DashboardTests(unittest.TestCase):\n    def setUp(self):\n        self.temp=tempfile.TemporaryDirectory();self.dbpatch=patch.object(auth,'DB_PATH',Path(self.temp.name)/'accounts.sqlite3');self.dbpatch.start()\n        auth.register('owner','a-long-test-password');auth.register('other','a-long-test-password')\n    def tearDown(self):self.dbpatch.stop();self.temp.cleanup()\n    def result(self,label,score):\n        return {'label':label,'confidence':score,'top_predictions':[{'label':label,'probability':score}], 'guidance':GUIDANCE[label],'stage':stage_info(label),'medical_disclaimer':'Educational only.'}\n    def test_real_aggregates_exclude_other_accounts(self):\n        a=reporting.save_report('owner',self.result('MEL',80))\n        b=reporting.save_report('owner',self.result('NEV',40))\n        c=reporting.save_report('other',self.result('SCC',100))\n        data=reporting.dashboard_data('owner')\n        self.assertEqual(data['stats']['total'],2);self.assertEqual(data['stats']['average'],60)\n        self.assertEqual(data['stats']['cancer_classes'],1);self.assertEqual(data['stats']['today'],2)\n        self.assertEqual(sum(m['count'] for m in data['months']),2)\n        self.assertEqual({r['report_id'] for r in data['reports']},{a,b})\n        self.assertNotIn(c,{r['report_id'] for r in data['reports']})\n        self.assertEqual(sum(r['count'] for r in data['frequencies']),2)\n    def test_empty_account_has_no_example_results(self):\n        data=reporting.dashboard_data('owner')\n        self.assertEqual(data['reports'],[]);self.assertEqual(data['stats']['total'],0)\n        self.assertEqual(len(data['months']),6)\n        self.assertTrue(all(month['height']==0 for month in data['months']))\n\nif __name__=='__main__':unittest.main()\n", 'tests/test_auth.py': "import sys\nimport tempfile\nimport time\nimport unittest\nfrom pathlib import Path\nfrom unittest.mock import patch\nfrom fastapi import HTTPException\nfrom starlette.requests import Request\nsys.path.insert(0,str(Path(__file__).resolve().parents[1]))\nfrom app import auth\nfrom app.guidance import stage_info\n\n\ndef request(token='',csrf='test',origin='http://localhost'):\n    return Request({'type':'http','headers':[(b'cookie',f'skinsight_session={token}; skinsight_csrf={csrf}'.encode()),(b'host',b'localhost'),(b'origin',origin.encode())]})\n\n\nclass AuthTests(unittest.TestCase):\n    def setUp(self):\n        self.temp=tempfile.TemporaryDirectory()\n        self.patch=patch.object(auth,'DB_PATH',Path(self.temp.name)/'accounts.sqlite3');self.patch.start()\n        auth.register('Teammate','a-unique-password')\n\n    def tearDown(self):\n        self.patch.stop();self.temp.cleanup()\n\n    def test_hashed_password_persistent_login_expiry_and_logout(self):\n        with auth.connect() as db:\n            row=db.execute('SELECT * FROM users').fetchone()\n            self.assertNotEqual(row['password_hash'],'a-unique-password')\n            self.assertEqual(row['username'],'teammate')\n        token=auth.login('TEAMMATE','a-unique-password','local')\n        self.assertEqual(auth.require_user(request(token)),'teammate')\n        auth.logout(request(token));self.assertIsNone(auth.current_user(request(token)))\n        token=auth.login('teammate','a-unique-password','local')\n        with patch.object(auth.time,'time',return_value=time.time()+auth.SESSION_SECONDS+1):\n            self.assertIsNone(auth.current_user(request(token)))\n\n    def test_bad_credentials_rate_limit_and_duplicate(self):\n        with self.assertRaises(ValueError):auth.register('TEAMMATE','another-password')\n        for _ in range(10):\n            with self.assertRaisesRegex(ValueError,'Incorrect'):auth.login('teammate','wrong-password','bad')\n        with self.assertRaisesRegex(ValueError,'Too many'):auth.login('teammate','a-unique-password','bad')\n        with self.assertRaises(ValueError):auth.register('short','short')\n\n    def test_csrf_checks_token_and_origin(self):\n        auth.check_csrf(request(),'test')\n        with self.assertRaises(HTTPException):auth.check_csrf(request(),'wrong')\n        with self.assertRaises(HTTPException):auth.check_csrf(request(origin='https://attacker.example'),'test')\n\n    def test_stage_is_never_assigned_from_prediction(self):\n        self.assertIn('Not determinable',stage_info('MEL')['status'])\n        self.assertTrue(stage_info('MEL')['education'])\n        for label in ['BCC','SCC']:self.assertIn('Not determinable',stage_info(label)['status'])\n        self.assertIn('Not applicable',stage_info('NEV')['status'])\n\nif __name__=='__main__':unittest.main()\n", 'tests/test_external.py': "import hashlib\nimport sys\nimport tempfile\nimport unittest\nfrom pathlib import Path\nimport pandas as pd\nimport torch\nfrom PIL import Image\nsys.path.insert(0, str(Path(__file__).resolve().parents[1] / 'ml'))\nfrom external_data import audit_external\nfrom calibrate import fit_temperature\n\nclass ExternalTests(unittest.TestCase):\n    def fixture(self, root):\n        paths=[]\n        for i in range(4):\n            image=root/f'{i}.png';Image.new('RGB',(64,64),(i*30,80,90)).save(image)\n            paths.append(dict(image_path=str(image),label='NEV',source='source',original_id=str(i),\n                              patient_id=f'p{i}',lesion_id=f'l{i}',image_sha256=hashlib.sha256(image.read_bytes()).hexdigest()))\n        for name,row in zip(['train','validation','test'],paths[:3]):\n            pd.DataFrame([row]).to_csv(root/f'{name}_split.csv',index=False)\n        pd.DataFrame([paths[3]]).to_csv(root/'external.csv',index=False)\n        return paths\n\n    def test_accepts_novel_images_and_rejects_old_test_and_calibration(self):\n        with tempfile.TemporaryDirectory() as temp:\n            root=Path(temp);rows=self.fixture(root)\n            _,audit=audit_external(root/'external.csv',root)\n            self.assertFalse(audit['known_group_overlap'])\n            with self.assertRaisesRegex(ValueError,'overlaps'):\n                audit_external(root/'test_split.csv',root)\n            pd.DataFrame([rows[3]]).to_csv(root/'calibration.csv',index=False)\n            with self.assertRaisesRegex(ValueError,'overlaps'):\n                audit_external(root/'external.csv',root,root/'calibration.csv')\n\n    def test_rejects_changed_image_and_patient_overlap(self):\n        with tempfile.TemporaryDirectory() as temp:\n            root=Path(temp);rows=self.fixture(root)\n            rows[3]['patient_id']=rows[0]['patient_id']\n            pd.DataFrame([rows[3]]).to_csv(root/'external.csv',index=False)\n            with self.assertRaisesRegex(ValueError,'overlaps'):\n                audit_external(root/'external.csv',root)\n            Image.new('RGB',(64,64),'red').save(root/'3.png')\n            with self.assertRaisesRegex(ValueError,'checksum'):\n                audit_external(root/'external.csv',root)\n\n    def test_calibration_cli_and_external_test_overlap_guard(self):\n        import subprocess,os,json\n        from model import create_model,CLASS_NAMES\n        with tempfile.TemporaryDirectory() as temp:\n            root=Path(temp);self.fixture(root)\n            checkpoint=root/'model.pt'\n            torch.save(dict(model_state=create_model(pretrained=False).state_dict(),\n                            architecture='mobilenet_v3_small',class_names=CLASS_NAMES,\n                            image_size=64),checkpoint)\n            base=Path(__file__).resolve().parents[1]/'ml'\n            env=os.environ.copy();env.update(OMP_NUM_THREADS='1',MKL_NUM_THREADS='1',NO_ALBUMENTATIONS_UPDATE='1')\n            calibrated=root/'calibrated.pt'\n            result=subprocess.run([sys.executable,str(base/'calibrate.py'),\n                '--manifest',str(root/'external.csv'),'--checkpoint',str(checkpoint),\n                '--reference-dir',str(root),'--output-checkpoint',str(calibrated)],\n                env=env,text=True,capture_output=True,timeout=60)\n            self.assertEqual(result.returncode,0,result.stdout+result.stderr)\n            self.assertTrue(calibrated.with_suffix('.calibration.csv').exists())\n            from app import inference\n            from unittest.mock import patch\n            with patch.object(inference,'CHECKPOINT',calibrated):\n                predictor=inference.Predictor()\n            self.assertGreater(predictor.temperature,0)\n            result=subprocess.run([sys.executable,str(base/'evaluate_external.py'),\n                '--manifest',str(root/'external.csv'),'--checkpoint',str(calibrated),\n                '--reference-dir',str(root),'--output',str(root/'report.json')],\n                env=env,text=True,capture_output=True,timeout=60)\n            self.assertNotEqual(result.returncode,0)\n            self.assertIn('overlaps',result.stderr)\n            self.assertFalse((root/'report.json').exists())\n\n    def test_temperature_reduces_nll_without_changing_classes(self):\n        logits=torch.tensor([[9.,0.],[9.,0.],[0.,9.],[0.,9.]])\n        labels=torch.tensor([0,1,1,0])\n        temperature,before,after=fit_temperature(logits,labels)\n        self.assertGreater(temperature,1)\n        self.assertLess(after,before)\n        self.assertTrue(torch.equal(logits.argmax(1),(logits/temperature).argmax(1)))\n\nif __name__ == '__main__':unittest.main()\n", 'tests/test_reports.py': "import os\nimport sqlite3\nimport sys\nimport tempfile\nimport unittest\nfrom pathlib import Path\nfrom unittest.mock import patch\nsys.path.insert(0,str(Path(__file__).resolve().parents[1]))\nfrom app import auth,reporting\nfrom app.guidance import GUIDANCE,stage_info\n\nclass ReportTests(unittest.TestCase):\n    def setUp(self):\n        self.temp=tempfile.TemporaryDirectory();self.dbpatch=patch.object(auth,'DB_PATH',Path(self.temp.name)/'accounts.sqlite3');self.dbpatch.start()\n        auth.register('owner','long-test-password','owner@example.com');auth.register('other','other-test-password','other@example.com')\n        self.result={'label':'MEL','confidence':80.,'top_predictions':[{'label':'MEL','probability':80.}], 'guidance':GUIDANCE['MEL'],'stage':stage_info('MEL'),'medical_disclaimer':'Educational screening, not a medical diagnosis.'}\n    def tearDown(self):self.dbpatch.stop();self.temp.cleanup()\n\n    def test_private_pdf_and_direct_delivery_to_registered_email(self):\n        report_id=reporting.save_report('owner',self.result)\n        self.assertIsNone(reporting.get_report('other',report_id))\n        data=reporting.get_report('owner',report_id);pdf=reporting.make_pdf(reporting.report_text(data))\n        self.assertTrue(pdf.startswith(b'%PDF-1.4'));self.assertIn(b'Doctor guidance',pdf);self.assertIn(b'Not determinable',pdf)\n        with patch.object(reporting,'send_mail',return_value='accepted') as sender:\n            reporting.deliver_report('owner',report_id)\n            args=sender.call_args.args;self.assertEqual(args[0],'owner@example.com');self.assertIn('dermatologist',args[2]);self.assertTrue(args[3].startswith(b'%PDF'))\n        self.assertEqual(reporting.get_report('owner',report_id)['email_status'],'accepted')\n\n    def test_email_edit_validation_and_missing_email(self):\n        auth.set_email('owner','new@example.com')\n        self.assertEqual(auth.email_profile('owner')['email'],'new@example.com')\n        with self.assertRaises(ValueError):auth.set_email('owner','evil\\r\\nBcc: other@example.com')\n        with self.assertRaises(ValueError):auth.set_email('owner','other@example.com')\n        auth.register('legacy','long-test-password')\n        report_id=reporting.save_report('legacy',self.result)\n        with patch.object(reporting,'send_mail') as sender:\n            reporting.deliver_report('legacy',report_id);sender.assert_not_called()\n        self.assertEqual(reporting.get_report('legacy',report_id)['email_status'],'email_required')\n\n    def test_gmail_tls_attachment_and_failure(self):\n        env={'SKINSIGHT_SMTP_HOST':'smtp.gmail.com','SKINSIGHT_SMTP_PORT':'465','SKINSIGHT_SMTP_USER':'sender@example.com','SKINSIGHT_SMTP_PASSWORD':'fake-test-only','SKINSIGHT_MAIL_FROM':'sender@example.com'}\n        with patch.dict(os.environ,env),patch.object(reporting.smtplib,'SMTP_SSL') as smtp:\n            smtp.return_value.__enter__.return_value.send_message.return_value={}\n            self.assertEqual(reporting.send_mail('owner@example.com','Report','Doctor guidance',b'%PDF-1.4'),'accepted')\n            server=smtp.return_value.__enter__.return_value\n            server.login.assert_called_once();message=server.send_message.call_args.args[0]\n            self.assertEqual(message['To'],'owner@example.com');self.assertEqual(len(list(message.iter_attachments())),1)\n            self.assertEqual(smtp.call_args.kwargs['timeout'],8)\n            self.assertTrue(smtp.call_args.kwargs['context'].check_hostname)\n        with patch.dict(os.environ,env),patch.object(reporting.smtplib,'SMTP_SSL',side_effect=OSError('offline')):\n            self.assertEqual(reporting.send_mail('owner@example.com','Report','test'),'failed')\n        with patch.dict(os.environ,env),patch.object(reporting.smtplib,'SMTP_SSL',side_effect=reporting.smtplib.SMTPAuthenticationError(535,b'test error')):\n            self.assertEqual(reporting.send_mail('owner@example.com','Report','test'),'auth_failed')\n            self.assertIn('App Password',reporting.delivery_message('auth_failed'))\n        with patch.dict(os.environ,{},clear=True):self.assertEqual(reporting.send_mail('owner@example.com','Report','test'),'not_configured')\n\n    def test_migrates_existing_account_database(self):\n        path=Path(self.temp.name)/'legacy.sqlite3'\n        with sqlite3.connect(path) as db:\n            db.execute('CREATE TABLE users (id INTEGER PRIMARY KEY, username TEXT UNIQUE NOT NULL, salt TEXT NOT NULL, password_hash TEXT NOT NULL)')\n        with patch.object(auth,'DB_PATH',path):\n            auth.register('legacy','long-test-password','legacy@example.com')\n            self.assertEqual(auth.email_profile('legacy')['email'],'legacy@example.com')\n\nif __name__=='__main__':unittest.main()\n", 'app/static/app.js': 'const input = document.getElementById(\'fileInput\');\nconst chooseBtn = document.getElementById(\'chooseBtn\');\nconst analyzeBtn = document.getElementById(\'analyzeBtn\');\nconst preview = document.getElementById(\'preview\');\nconst placeholder = document.getElementById(\'placeholder\');\nconst fileName = document.getElementById(\'fileName\');\nconst error = document.getElementById(\'error\');\nconst dropzone = document.getElementById(\'dropzone\');\nconst overlay = document.getElementById(\'loadingOverlay\');\nlet imageVersion = 0;\nlet selectedFile = null;\nlet busy = false;\n\nif (input) {\n  const choose = () => { if (!busy) input.click(); };\n  chooseBtn.addEventListener(\'click\', choose);\n  dropzone.addEventListener(\'click\', choose);\n  dropzone.addEventListener(\'keydown\', event => {\n    if (event.key === \'Enter\' || event.key === \' \') { event.preventDefault(); choose(); }\n  });\n  input.addEventListener(\'change\', () => selectFile(input.files?.[0]));\n  [\'dragenter\', \'dragover\'].forEach(name => dropzone.addEventListener(name, event => {\n    event.preventDefault(); if (!busy) dropzone.classList.add(\'dragging\');\n  }));\n  [\'dragleave\', \'drop\'].forEach(name => dropzone.addEventListener(name, event => {\n    event.preventDefault(); dropzone.classList.remove(\'dragging\');\n  }));\n  dropzone.addEventListener(\'drop\', event => { if (!busy) selectFile(event.dataTransfer.files?.[0]); });\n  analyzeBtn.addEventListener(\'click\', analyze);\n}\n\nfunction selectFile(file) {\n  if (!file || busy) return;\n  imageVersion++;\n  selectedFile = null;\n  analyzeBtn.disabled = true;\n  clearResult();\n  error.hidden = true;\n  if (preview.src.startsWith(\'blob:\')) URL.revokeObjectURL(preview.src);\n  preview.removeAttribute(\'src\'); preview.hidden = true; placeholder.hidden = false; fileName.textContent = \'\';\n  if (![\'image/jpeg\', \'image/png\', \'image/webp\'].includes(file.type)) { showError(\'Choose a JPG, PNG or WEBP skin photo.\'); return; }\n  if (file.size > 10 * 1024 * 1024) { showError(\'Please choose an image no larger than 10 MB.\'); return; }\n  selectedFile = file;\n  preview.src = URL.createObjectURL(file); preview.hidden = false; placeholder.hidden = true;\n  fileName.textContent = `${file.name} · ${(file.size / 1024 / 1024).toFixed(2)} MB`;\n  analyzeBtn.disabled = false;\n}\n\nasync function analyze() {\n  if (!selectedFile || busy) return;\n  busy = true;\n  const version = imageVersion;\n  clearResult(); error.hidden = true;\n  analyzeBtn.disabled = true; chooseBtn.disabled = true;\n  analyzeBtn.textContent = \'Analyzing…\'; overlay.hidden = false;\n  document.querySelector(\'.analysis-grid\').setAttribute(\'aria-busy\', \'true\');\n  const form = new FormData(); form.append(\'image\', selectedFile);\n  try {\n    const response = await fetch(\'/api/predict\', {method: \'POST\', body: form, headers: {\'X-CSRF-Token\': document.querySelector(\'meta[name="csrf-token"]\').content}});\n    const data = await response.json();\n    if (version !== imageVersion) return;\n    if (!response.ok) throw new Error(data.detail || \'Analysis failed. Please try again.\');\n    renderResult(data); watchEmail(data.email_status_url, version);\n  } catch (failure) {\n    if (version === imageVersion) showError(failure.message);\n  } finally {\n    busy = false; overlay.hidden = true;\n    document.querySelector(\'.analysis-grid\').removeAttribute(\'aria-busy\');\n    analyzeBtn.disabled = !selectedFile; chooseBtn.disabled = false;\n    analyzeBtn.textContent = \'Analyze image →\';\n  }\n}\n\nfunction renderResult(data) {\n  document.getElementById(\'downloadReport\').href = data.report_url;\n  document.getElementById(\'emailStatus\').textContent = \'Checking email status…\';\n  document.getElementById(\'emptyResult\').hidden = true;\n  document.getElementById(\'result\').hidden = false;\n  document.getElementById(\'diagnosis\').textContent = data.guidance.name;\n  document.getElementById(\'confidence\').textContent = `${data.confidence.toFixed(1)}%`;\n  document.getElementById(\'risk\').textContent = data.guidance.risk;\n  document.getElementById(\'message\').textContent = data.guidance.message;\n  document.getElementById(\'doctorText\').textContent = data.guidance.urgency;\n  document.getElementById(\'stageStatus\').textContent = data.stage.status;\n  document.getElementById(\'stageText\').textContent = data.stage.explanation;\n  const guide = document.getElementById(\'stageGuide\'); guide.replaceChildren();\n  (data.stage.education || []).forEach(item => {\n    const p = document.createElement(\'p\'); p.textContent = item; guide.appendChild(p);\n  });\n  document.getElementById(\'disclaimer\').textContent = data.medical_disclaimer;\n  const names = {ACK: \'Actinic keratosis\', BCC: \'Basal cell carcinoma\', MEL: \'Melanoma\', NEV: \'Melanocytic nevus\', SCC: \'Squamous cell carcinoma\', SEK: \'Benign / seborrheic keratosis\'};\n  const box = document.getElementById(\'probabilities\'); box.replaceChildren();\n  data.top_predictions.forEach(item => {\n    const row = document.createElement(\'div\'); row.className = \'prob-row\';\n    const label = document.createElement(\'div\'); label.className = \'prob-label\';\n    const name = document.createElement(\'span\'); name.textContent = names[item.label] || item.label;\n    const score = document.createElement(\'span\'); score.textContent = `${item.probability.toFixed(1)}%`;\n    label.append(name, score);\n    const track = document.createElement(\'div\'); track.className = \'track\';\n    const fill = document.createElement(\'div\'); fill.className = \'fill\'; fill.style.width = `${Math.max(0, Math.min(100, item.probability))}%`;\n    track.appendChild(fill); row.append(label, track); box.appendChild(row);\n  });\n}\n\nfunction clearResult() {\n  document.getElementById(\'result\').hidden = true;\n  document.getElementById(\'emptyResult\').hidden = false;\n}\nfunction showError(message) { clearResult(); error.textContent = message; error.hidden = false; }\n\nasync function watchEmail(url, version) {\n  const messages = {auth_failed: \'Gmail rejected the sender login. Check the saved sender address and Google App Password.\', recipient_refused: \'Check your recipient email address in Email settings.\', sender_refused: \'Gmail rejected the sending address. Check sender configuration.\', tls_failed: \'Secure Gmail connection failed. Check computer date/time and network.\', accepted: \'Gmail accepted your report email. Check your inbox and spam folder.\', failed: \'Email could not be sent. You can still download the PDF. Check the sending Gmail configuration.\', not_configured: \'Email sending is not configured. Your PDF is ready to download.\', email_required: \'Add a report email in Email settings to receive future reports.\'};\n  for (let attempt = 0; attempt < 25; attempt++) {\n    if (version !== imageVersion) return;\n    try {\n      const response = await fetch(url, {cache: \'no-store\'});\n      if (!response.ok) throw new Error();\n      const data = await response.json();\n      if (version !== imageVersion) return;\n      if (data.email_status !== \'pending\') {\n        document.getElementById(\'emailStatus\').textContent = messages[data.email_status] || \'Your report is ready to download.\';\n        return;\n      }\n    } catch (_) {\n      if (version === imageVersion) document.getElementById(\'emailStatus\').textContent = \'Email status unavailable. Your PDF can still be downloaded.\';\n      return;\n    }\n    await new Promise(resolve => setTimeout(resolve, 1000));\n  }\n  if (version === imageVersion) document.getElementById(\'emailStatus\').textContent = \'Email is still processing. Your PDF is ready to download.\';\n}\nif (window.savedReport) { renderResult(window.savedReport); watchEmail(window.savedReport.email_status_url, imageVersion); }\n', 'app/static/styles.css': ':root{color-scheme:dark;--bg:#091321;--surface:#101e30;--surface2:#14253a;--text:#edf3ff;--muted:#a0b2ca;--line:#27394e;--blue:#477dff;--cyan:#28c3d7;--shadow:0 22px 70px #02091155;--radius:20px}\nhtml[data-theme=light]{color-scheme:light;--bg:#f4f7fb;--surface:#fff;--surface2:#eef3fa;--text:#12233b;--muted:#51657e;--line:#d5dfec;--blue:#245ee9;--cyan:#078da4;--shadow:0 16px 45px #14233b12}\n*{box-sizing:border-box}html{scroll-behavior:smooth;scroll-padding-top:90px}body{margin:0;background:var(--bg);color:var(--text);font-family:Inter,"Segoe UI",Arial,sans-serif;line-height:1.6}a{color:inherit;text-decoration:none}button,input{font:inherit}button,a,input,summary{ -webkit-tap-highlight-color:transparent}button{cursor:pointer}button:disabled{opacity:.45;cursor:not-allowed}a:focus-visible,button:focus-visible,summary:focus-visible,[tabindex]:focus-visible{outline:3px solid var(--cyan);outline-offset:4px}[hidden]{display:none!important}h1,h2,h3,h4,p{margin:0}h1{font-size:clamp(36px,5vw,64px);line-height:1.08;letter-spacing:-.055em;font-weight:750}h2{font-size:clamp(25px,3vw,36px);line-height:1.2;letter-spacing:-.035em}h3{font-size:18px;line-height:1.3}p{margin:12px 0;color:var(--muted)}strong,b{font-weight:650}small{font-size:12px}.container{max-width:1180px;margin:0 auto;padding-left:28px;padding-right:28px}.narrow{max-width:840px}.section{padding-top:70px;padding-bottom:70px}.lead{font-size:17px;line-height:1.7;max-width:600px}.eyebrow{font-size:11px;font-weight:750;letter-spacing:.17em;color:var(--cyan);margin-bottom:20px}.tiny-dot{display:inline-block;width:6px;height:6px;background:var(--cyan);border-radius:50%;vertical-align:middle;margin-right:8px}.gradient-text{background:linear-gradient(110deg,var(--blue),var(--cyan));background-clip:text;-webkit-background-clip:text;color:transparent}.muted{color:var(--muted)}.small{font-size:13px}.text-link{color:var(--cyan);font-weight:600}.text-link:hover{color:var(--blue)}\n.site-header{position:sticky;top:0;z-index:30;background:var(--bg);border-bottom:1px solid var(--line)}.nav-wrap{max-width:1240px;padding:18px 28px;margin:auto;display:flex;align-items:center;gap:25px}.brand{display:inline-flex;align-items:center;gap:7px;font-size:19px;font-weight:750;white-space:nowrap;letter-spacing:-.5px}.brand-ai{color:var(--cyan);font-weight:600}.brand-mark{height:32px;width:32px;border-radius:10px;background:linear-gradient(140deg,#467bff,#14aebd);color:white;display:inline-flex;align-items:center;justify-content:center;font-size:20px;position:relative;margin-right:3px;box-shadow:0 4px 18px #397fff24}.brand-mark span{position:absolute;right:4px;top:1px;font-size:13px}.nav-wrap nav{display:flex;align-items:center;gap:22px;margin-left:auto}.nav-wrap nav a{color:var(--muted);font-size:13px;font-weight:600;padding:6px 0;border-bottom:2px solid transparent}.nav-wrap nav a:hover,.nav-wrap nav a[aria-current]{color:var(--text);border-color:var(--cyan)}.nav-actions{display:flex;gap:12px;align-items:center}.icon-button{display:inline-flex;align-items:center;justify-content:center;width:34px;height:34px;border:1px solid var(--line);border-radius:50%;background:var(--surface);color:var(--text);font-size:17px}.avatar{display:grid;place-items:center;width:32px;height:32px;background:linear-gradient(120deg,var(--blue),var(--cyan));border-radius:50%;color:white;font-size:13px;font-weight:700}.login-link{font-size:13px;color:var(--muted)}.mobile-nav{display:none}.btn{display:inline-flex;align-items:center;justify-content:center;gap:14px;padding:13px 22px;background:var(--blue);color:white;border:1px solid transparent;border-radius:10px;font-size:14px;line-height:1.3;font-weight:650;transition:transform .15s,background .15s}.btn:hover{background:#3368e3;transform:translateY(-1px)}.btn span{font-size:18px}.btn-outline{background:transparent;color:var(--text);border-color:var(--line)}.btn-outline:hover{background:var(--surface2)}.btn-small{padding:8px 14px;border-radius:8px;font-size:12px}.full-width{width:100%}.actions{display:flex;gap:12px;flex-wrap:wrap;margin-top:28px}\n.hero{display:grid;grid-template-columns:1.03fr 1fr;align-items:center;gap:50px;min-height:620px;padding-top:75px;padding-bottom:75px}.hero .lead{margin-top:25px;max-width:490px}.hero-tags{display:flex;gap:20px;font-size:11px;color:var(--muted);margin-top:24px;flex-wrap:wrap}.hero-visual{position:relative;height:420px;background:radial-gradient(ellipse at 50% 50%,#137e9736,transparent 70%),var(--surface);border:1px solid var(--line);border-radius:32px;overflow:hidden}.visual-grid{position:absolute;inset:0;background-image:linear-gradient(#3994b10c 1px,transparent 1px),linear-gradient(90deg,#3994b10c 1px,transparent 1px);background-size:30px 30px;mask-image:radial-gradient(ellipse,black,transparent)}.scan-orbit{position:absolute;border:1px solid #398fd344;border-radius:50%;left:50%;top:50%;transform:translate(-50%,-50%)}.orbit-one{width:330px;height:330px}.orbit-two{width:275px;height:275px;border-style:dashed}.scan-core{position:absolute;width:215px;height:215px;border-radius:45px;top:50%;left:50%;transform:translate(-50%,-50%) rotate(-10deg);border:1px solid #64cdd187;background:radial-gradient(circle at 70% 20%,#99d9d036,transparent),linear-gradient(145deg,#23587a,#182c49);box-shadow:0 0 60px #22afc827;overflow:hidden}.cell{position:absolute;background:#3b90a088;border:1px solid #80cbd587;border-radius:42%}.c1{width:85px;height:85px;top:25px;left:15px}.c2{width:105px;height:100px;right:-10px;top:40px}.c3{width:80px;height:90px;top:118px;left:26px}.c4{width:90px;height:105px;bottom:-13px;right:1px}.c5{width:55px;height:50px;left:91px;top:94px;background:#4475ae}.scan-line{height:1px;background:#91eaff;box-shadow:0 0 12px #60daee;position:absolute;left:0;right:0;top:52%}.float-card{position:absolute;display:flex;align-items:center;gap:12px;background:var(--surface);border:1px solid var(--line);padding:13px 17px;border-radius:12px;box-shadow:var(--shadow);font-size:12px}.float-card strong{display:block}.float-card small{display:block;color:var(--muted);font-size:10px}.top-float{top:42px;right:18px}.bottom-float{bottom:40px;left:20px}.float-icon{font-size:25px;color:var(--cyan)}.float-icon.green{color:#64d2a4}.visual-caption{position:absolute;bottom:13px;right:20px;color:var(--muted);font-size:8px;letter-spacing:2px}.trust-strip{display:grid;grid-template-columns:repeat(4,1fr);border-top:1px solid var(--line);border-bottom:1px solid var(--line);padding-top:24px;padding-bottom:24px}.trust-strip div{padding-left:25px;border-left:1px solid var(--line)}.trust-strip div:first-child{padding-left:0;border:0}.trust-strip strong{font-size:23px;display:block}.trust-strip span{font-size:12px;color:var(--muted)}.section-heading{margin-bottom:32px;max-width:630px}.section-heading p:not(.eyebrow){font-size:15px}.three-grid{display:grid;grid-template-columns:repeat(3,1fr);gap:20px}.two-grid{display:grid;grid-template-columns:repeat(2,1fr);gap:24px}.feature-card,.panel,.stat-card{background:var(--surface);border:1px solid var(--line);border-radius:var(--radius);padding:28px}.step-number{font-size:13px;color:var(--cyan);display:inline-block;padding:7px 11px;background:var(--surface2);border-radius:8px;margin-bottom:25px}.feature-card p{font-size:14px;line-height:1.8}.callout{display:flex;align-items:center;justify-content:space-between;gap:35px;margin-bottom:80px;margin-top:35px;padding:40px;border:1px solid var(--line);border-radius:var(--radius);background:linear-gradient(100deg,var(--surface),var(--surface2))}.callout p{max-width:620px}.callout .btn{flex-shrink:0}.page-heading{padding-top:75px;padding-bottom:20px}.page-heading .lead{margin-top:23px}.panel p{font-size:14px;line-height:1.8}.section-icon{display:inline-grid;place-items:center;background:var(--surface2);color:var(--cyan);height:42px;width:42px;border-radius:12px;font-size:23px;flex-shrink:0;margin-bottom:14px}.panel h2{margin-bottom:17px}.model-path{display:flex;align-items:center;gap:10px;margin:24px 0;font-size:12px;flex-wrap:wrap}.model-path span{padding:7px 10px;background:var(--surface2);border:1px solid var(--line);border-radius:7px}.model-path b{color:var(--cyan)}.dataset-row{display:flex;align-items:center;justify-content:space-between;padding:18px 0;border-bottom:1px solid var(--line)}.dataset-row p{font-size:12px;margin:4px 0}.dataset-row>strong{font-size:22px}.notice{display:flex;align-items:flex-start;gap:20px;margin-top:25px}.notice .section-icon{margin:0}.notice p{margin-bottom:0}.stage-timeline{display:grid;grid-template-columns:repeat(4,1fr);gap:20px}.stage-timeline article{padding:24px;background:var(--surface);border:1px solid var(--line);border-radius:16px}.stage-timeline article>span{display:inline-grid;place-items:center;width:48px;height:48px;border:1px solid var(--cyan);border-radius:50%;font-size:18px;color:var(--cyan);margin-bottom:24px}.stage-timeline p{font-size:14px}.faq-list{margin-top:28px;margin-bottom:70px}.faq-item{border-bottom:1px solid var(--line);padding:20px 0}.faq-item summary{list-style:none;font-size:17px;font-weight:600;cursor:pointer;display:flex;justify-content:space-between;gap:20px}.faq-item summary::-webkit-details-marker{display:none}.faq-item summary>span{color:var(--cyan);font-size:22px}.faq-item[open] summary>span{transform:rotate(45deg)}.faq-item p{max-width:710px;font-size:15px;padding-right:25px}\n.analysis-heading{text-align:center;padding-top:60px;padding-bottom:40px}.analysis-heading h1{font-size:clamp(33px,4vw,47px)}.analysis-heading .lead{margin:20px auto;font-size:15px}.steps{display:flex;gap:16px;justify-content:center;align-items:center;font-size:11px;color:var(--muted);margin-top:25px}.steps span{display:flex;align-items:center;gap:7px}.steps b{display:grid;place-items:center;width:23px;height:23px;border-radius:50%;background:var(--surface2)}.steps .active b{background:var(--blue);color:white}.steps i{font-style:normal;color:var(--line)}.analysis-grid{display:grid;grid-template-columns:1fr 1.08fr;align-items:start;gap:24px;margin-bottom:90px}.panel-title{display:flex;justify-content:space-between;align-items:center;gap:12px;margin-bottom:20px}.panel-title h2{font-size:21px;margin:0}.status-pill{display:inline-block;padding:4px 9px;border:1px solid var(--line);border-radius:25px;font-size:10px;color:var(--muted);white-space:nowrap}.status-pill.available{color:#59d5b0;background:#25a48912;border-color:#25a48940}.dropzone{display:flex;align-items:center;justify-content:center;width:100%;min-height:290px;border:1px dashed #5b7290;border-radius:14px;background:var(--surface2);overflow:hidden;text-align:center;cursor:pointer}.dropzone.dragging{border-color:var(--cyan);background:#249aa624}.dropzone>img{display:block;width:100%;max-height:370px;object-fit:contain}.dropzone p{font-size:12px;margin:8px 0 18px}.upload-icon{display:grid;place-items:center;width:58px;height:58px;border:1px solid #477dff70;border-radius:17px;background:linear-gradient(140deg,#477dff,#17b5c9);color:white;font-size:34px;margin:0 auto 20px;box-shadow:0 0 0 9px #477dff10}.file-hint{font-size:10px;color:var(--muted);padding:5px 10px;border:1px solid var(--line);border-radius:20px}.photo-tips{display:grid;grid-template-columns:repeat(3,1fr);gap:8px;margin-top:12px}.photo-tips>div{background:var(--bg);border-radius:8px;padding:10px 8px;display:grid;grid-template-columns:24px 1fr;align-items:center}.photo-tips span{grid-row:span 2;font-size:18px;color:var(--cyan)}.photo-tips b{font-size:9px}.photo-tips small{font-size:8px;color:var(--muted)}.upload-actions{display:flex;gap:10px;margin-top:14px}.upload-actions .btn{flex:1;font-size:12px;padding:12px 8px}.privacy-note{font-size:10px!important;text-align:center;margin-top:14px!important}.result-panel{min-height:475px}.empty-state{display:flex;align-items:center;justify-content:center;flex-direction:column;text-align:center;min-height:380px}.empty-symbol{display:grid;place-items:center;border:1px solid var(--line);border-radius:50%;width:80px;height:80px;font-size:39px;color:var(--cyan);margin-bottom:22px;background:var(--surface2)}.empty-state p{font-size:13px}.empty-state.compact{min-height:290px}.result-heading{display:flex;justify-content:space-between;gap:20px;align-items:flex-start}.result-heading h2{font-size:27px;line-height:1.15;margin-bottom:13px}.result-heading .eyebrow{font-size:9px;margin-bottom:10px}.result-tag{font-size:10px;padding:5px 9px;border:1px solid var(--line);border-radius:20px;color:var(--muted)}.score{text-align:right;padding-top:20px}.score strong{font-size:28px;color:var(--cyan)}.score span{display:block;font-size:10px;color:var(--muted)}.result-description{padding-bottom:18px;border-bottom:1px solid var(--line);font-size:13px!important}.guidance-card{display:flex;gap:13px;background:#1b9eaf13;border:1px solid #278ca532;border-radius:12px;padding:17px;margin-top:20px}.guidance-card .section-icon{width:25px;height:25px;font-size:19px;background:transparent;margin:0}.guidance-card h3{font-size:13px}.guidance-card p{font-size:12px;margin:7px 0 0}.subheading{font-size:14px;margin:24px 0 12px}.prob-row{margin-bottom:14px}.prob-label{display:flex;justify-content:space-between;font-size:11px;margin-bottom:6px}.track{height:5px;background:var(--surface2);border-radius:5px;overflow:hidden}.fill,.track>span{display:block;height:100%;background:linear-gradient(90deg,var(--blue),var(--cyan));border-radius:5px}.report-actions{display:flex;gap:9px;flex-wrap:wrap;margin-top:26px}.report-actions .btn{font-size:11px;padding:11px 13px}.stage-result{margin-top:20px;border:1px solid var(--line);border-radius:12px;padding:16px}.stage-result summary{cursor:pointer;font-size:13px;font-weight:600;display:flex;justify-content:space-between;list-style:none}.stage-result p,.stage-result a{font-size:12px}.disclaimer{font-size:10px!important;line-height:1.7!important;border-top:1px solid var(--line);padding-top:15px;margin-top:24px!important}.error{background:#e54b4b12;border:1px solid #e54b4b55;border-radius:10px;padding:12px;color:#ff9b9b!important;font-size:13px}.loading-overlay{position:fixed;inset:0;z-index:80;background:#030c19bb;backdrop-filter:blur(6px);display:grid;place-items:center;padding:20px}.loading-card{max-width:420px;width:100%;padding:40px;border:1px solid var(--line);border-radius:24px;text-align:center;background:var(--surface);box-shadow:var(--shadow)}.loading-card h2{font-size:23px}.loading-card p,.loading-card small{font-size:12px;color:var(--muted)}.loading-card .eyebrow{font-size:9px;color:var(--cyan)}.loading-orbit{height:72px;width:72px;display:grid;place-items:center;border-radius:50%;border:1px dashed var(--cyan);color:var(--blue);font-size:26px;margin:0 auto 25px;animation:breathe 2s infinite}.loading-track{height:4px;background:var(--surface2);border-radius:4px;overflow:hidden;margin:25px 0 17px}.loading-track span{display:block;height:100%;width:35%;background:linear-gradient(90deg,var(--blue),var(--cyan));animation:loading 1.6s infinite}@keyframes loading{from{transform:translateX(-120%)}to{transform:translateX(400%)}}@keyframes breathe{50%{box-shadow:0 0 0 10px #20b5c210}}\n.auth-layout{display:grid;grid-template-columns:1fr 1fr;padding:0;max-width:920px;border:1px solid var(--line);border-radius:26px;overflow:hidden;margin-top:70px;margin-bottom:80px;box-shadow:var(--shadow)}.auth-story{padding:45px;background:linear-gradient(145deg,#244fea,#087f97);color:white;position:relative;overflow:hidden}.auth-story h1{font-size:43px;letter-spacing:-.045em}.auth-story .eyebrow{color:#c1e7ff;font-size:10px}.auth-story p{color:#d0e6fb;font-size:14px}.auth-story h1 span{color:#bdeeff}.auth-art{height:140px;position:relative;display:grid;place-items:center}.auth-art>span{font-size:47px;color:#c6f5ff;position:relative}.auth-ring{height:120px;width:120px;border:1px solid #b4edff55;border-radius:50%;position:absolute;box-shadow:0 0 0 16px #b4edff13,0 0 0 30px #b4edff08}.auth-note{margin-top:20px}.auth-note b{font-size:13px}.auth-note p{font-size:11px}.auth-form{padding:45px;background:var(--surface)}.auth-form h2{font-size:32px}.auth-form .eyebrow{font-size:9px}.auth-form p{font-size:13px}.auth-form form{margin-top:25px}.auth-form label,.account-panel label{display:block;font-size:12px;font-weight:600;margin-bottom:7px}.auth-input{display:block;width:100%;padding:12px 14px;margin-bottom:20px;border:1px solid var(--line);border-radius:9px;background:var(--bg);color:var(--text);font-size:13px;outline:none}.auth-input:focus{border-color:var(--blue);box-shadow:0 0 0 3px #477dff20}.auth-switch{margin-top:20px!important;text-align:center}.auth-switch a{color:var(--cyan)}.account-panel{max-width:650px;margin:30px auto 75px}.account-panel label{margin-top:28px}\n.dashboard-heading{display:flex;align-items:center;justify-content:space-between;padding-top:60px;padding-bottom:35px;gap:25px}.dashboard-heading h1{font-size:42px}.dashboard-heading .lead{font-size:14px}.stats-grid{display:grid;grid-template-columns:repeat(4,1fr);gap:18px;margin-bottom:26px}.stat-card{padding:20px;position:relative}.stat-icon{display:grid;place-items:center;width:34px;height:34px;background:#477dff21;color:#89acff;border-radius:10px;margin-bottom:16px;font-size:20px}.stat-icon.cyan{background:#26becb20;color:#58d3dd}.stat-icon.green{background:#29b68c20;color:#6cddb9}.stat-icon.amber{background:#d8a64620;color:#e7c479}.stat-card small{display:block;color:var(--muted);font-size:11px}.stat-card>strong{display:block;font-size:29px;margin-top:4px}.stat-card p{font-size:10px;margin:3px 0 0}.dashboard-charts{margin-bottom:25px}.bar-chart{display:flex;align-items:flex-end;gap:20px;height:190px;margin-top:20px}.bar-column{display:flex;flex:1;height:100%;align-items:center;flex-direction:column;justify-content:flex-end;gap:7px;min-width:0}.bar-column>b{font-size:11px;color:var(--muted)}.bar-track{height:135px;width:80%;display:flex;align-items:flex-end;border-bottom:1px solid var(--line)}.bar-track span{display:block;width:100%;background:linear-gradient(var(--blue),#315dcc);border-radius:5px 5px 0 0}.bar-column small{font-size:9px;color:var(--muted);white-space:nowrap}.frequency-list{display:flex;flex-direction:column;gap:18px;margin-top:25px}.frequency-list>div{display:flex;gap:15px;align-items:center}.category-label{width:40px;font-size:11px;color:var(--muted)}.frequency-list .track{flex:1;height:7px}.frequency-list b{width:24px;text-align:right;font-size:12px}.activity-panel{margin-bottom:25px}.activity-panel .panel-title p{font-size:12px;margin:6px 0}.table-scroll{overflow-x:auto}.activity-table{width:100%;border-collapse:collapse;font-size:12px;text-align:left}.activity-table th{padding:13px 15px 13px 0;text-transform:uppercase;letter-spacing:.08em;font-size:9px;color:var(--muted);font-weight:600;border-bottom:1px solid var(--line);white-space:nowrap}.activity-table td{padding:19px 18px 19px 0;border-bottom:1px solid var(--line)}.activity-table td:first-child a{font-size:13px;font-weight:650}.activity-table td small{display:block;font-size:10px;color:var(--muted);margin-top:4px}.activity-table td:nth-child(3){white-space:nowrap}.status-text{font-size:10px;color:var(--muted)}.download-link{display:block;color:var(--muted);font-size:10px;margin-top:5px}.dashboard-note{display:flex;justify-content:space-between;gap:20px;margin-bottom:70px}.dashboard-note p{font-size:11px;margin:0}.dashboard-note a{font-size:11px;flex-shrink:0}.report-view{max-width:760px;margin:25px auto 80px}\n.site-footer{border-top:1px solid var(--line);padding:45px 28px 20px;background:var(--surface)}.footer-grid{max-width:1124px;margin:0 auto;display:grid;grid-template-columns:1.6fr 1fr 1fr 1.4fr;gap:40px}.footer-grid h4{font-size:12px;margin-bottom:15px}.footer-grid>div>a:not(.brand){display:block;font-size:12px;color:var(--muted);margin:8px 0}.footer-grid p{font-size:12px;line-height:1.8}.footer-bottom{max-width:1124px;margin:30px auto 0;padding-top:17px;border-top:1px solid var(--line);display:flex;justify-content:space-between;gap:20px;font-size:10px;color:var(--muted)}\n@media(max-width:1050px){.nav-wrap{gap:16px}.nav-wrap nav{gap:14px}.nav-wrap nav a{font-size:12px}.nav-actions{gap:8px}.hero{gap:25px}.hero h1{font-size:48px}.hero-tags{gap:10px}.float-card{font-size:10px}.photo-tips b{font-size:8px}.analysis-grid{gap:17px}.panel{padding:22px}.stage-timeline{gap:14px}.stage-timeline article{padding:20px}}\n@media(max-width:820px){.nav-wrap{flex-wrap:wrap;padding:14px 20px}.mobile-nav{display:inline-flex;margin-left:auto}.nav-actions{margin-left:0}.nav-wrap nav{display:none;order:5;width:100%;flex-wrap:wrap;margin:0;padding:10px 0}.nav-wrap nav.open{display:flex}.hero{grid-template-columns:1fr;min-height:0;padding-top:55px;gap:35px}.hero-copy{max-width:620px}.hero-visual{height:360px}.trust-strip{grid-template-columns:repeat(2,1fr);row-gap:25px}.trust-strip div:nth-child(3){border:0;padding-left:0}.three-grid{grid-template-columns:1fr}.two-grid,.analysis-grid{grid-template-columns:1fr}.callout{flex-direction:column;align-items:flex-start;padding:28px}.stage-timeline{grid-template-columns:repeat(2,1fr)}.analysis-heading{padding-top:40px}.analysis-grid{max-width:650px}.result-panel{min-height:0}.auth-layout{margin:40px 25px;grid-template-columns:1fr 1.2fr}.auth-story{padding:30px}.auth-story h1{font-size:34px}.auth-form{padding:30px}.stats-grid{grid-template-columns:repeat(2,1fr)}.footer-grid{grid-template-columns:repeat(2,1fr);gap:25px}.report-view{margin-left:25px;margin-right:25px}.dashboard-heading h1{font-size:36px}.dashboard-note{flex-direction:column}.account-panel{margin-left:25px;margin-right:25px}}\n@media(max-width:540px){.container{padding-left:20px;padding-right:20px}.nav-actions .login-link{display:none}.brand{font-size:17px}.brand-mark{width:29px;height:29px}.nav-wrap{gap:10px;padding:13px 16px}.nav-actions{gap:7px}.btn-small{font-size:10px;padding:8px 10px}.hero h1{font-size:42px}.hero-visual{height:310px}.float-card{padding:10px;font-size:9px}.top-float{right:10px;top:25px}.bottom-float{left:10px;bottom:27px}.scan-core{width:190px;height:190px}.orbit-one{height:280px;width:280px}.orbit-two{height:240px;width:240px}.hero{padding-bottom:40px}.hero-tags{font-size:10px}.hero .lead{font-size:15px}.trust-strip strong{font-size:21px}.trust-strip span{font-size:10px}.section{padding-top:45px;padding-bottom:45px}.page-heading{padding-top:45px}.stage-timeline{grid-template-columns:1fr}.stage-timeline article{display:flex;gap:20px;align-items:flex-start}.stage-timeline article>span{flex-shrink:0;margin:0}.analysis-grid .panel{padding:19px}.analysis-grid{padding-left:15px;padding-right:15px}.photo-tips b{font-size:9px}.dropzone{min-height:245px}.photo-tips small{font-size:7px}.result-heading h2{font-size:23px}.score strong{font-size:24px}.auth-layout{display:block;margin:30px 18px}.auth-story{padding:28px}.auth-story h1{font-size:32px}.auth-art,.auth-note{display:none}.auth-form{padding:28px}.auth-story p{font-size:12px}.dashboard-heading{align-items:flex-start;flex-direction:column;padding-top:40px}.stats-grid{gap:12px}.stat-card{padding:17px}.stat-card small{font-size:10px}.footer-grid{grid-template-columns:1fr 1fr;font-size:11px}.footer-grid>div:first-child,.footer-grid>div:last-child{grid-column:span 2}.footer-bottom{flex-direction:column;gap:8px}.bar-chart{gap:12px}.bar-column small{font-size:8px}.panel-title{align-items:flex-start}.panel-title h2{font-size:19px}}\n@media(prefers-reduced-motion:reduce){html{scroll-behavior:auto}*,*::before,*::after{animation:none!important;transition:none!important}}\n', 'app/static/site.js': "(() => {\n  const root = document.documentElement;\n  try { root.dataset.theme = localStorage.getItem('skinsight-theme') || 'dark'; } catch (_) {}\n  const toggle = document.getElementById('themeToggle');\n  const updateLabel = () => toggle?.setAttribute('aria-label', `Switch to ${root.dataset.theme === 'dark' ? 'light' : 'dark'} theme`);\n  updateLabel();\n  toggle?.addEventListener('click', () => {\n    root.dataset.theme = root.dataset.theme === 'dark' ? 'light' : 'dark';\n    try { localStorage.setItem('skinsight-theme', root.dataset.theme); } catch (_) {}\n    updateLabel();\n  });\n  const nav = document.getElementById('mainNav');\n  document.getElementById('navToggle')?.addEventListener('click', event => {\n    const open = nav.classList.toggle('open');\n    event.currentTarget.setAttribute('aria-expanded', String(open));\n  });\n})();\n", 'app/templates/index.html': '{% extends \'base.html\' %}{% block title %}Analyze a photo · SkinSight AI{% endblock %}{% block content %}\n<section class="analysis-heading container"><p class="eyebrow"><span class="tiny-dot"></span> YOUR SKIN-PHOTO WORKSPACE</p><h1>One photo.<br/><span class="gradient-text">A clearer next step.</span></h1><p class="lead">Explore six lesion categories with MobileNetV3-Small.<br/>A screening result starts a conversation—it does not give a diagnosis.</p><div class="steps"><span class="active"><b>1</b> Upload</span><i>→</i><span><b>2</b> Analyze</span><i>→</i><span><b>3</b> Understand</span></div></section>\n<section class="container analysis-grid"><div class="panel upload-panel"><div class="panel-title"><h2>Your skin photo</h2><span class="status-pill {{ \'available\' if model_ready else \'\' }}">{{ \'Model ready\' if model_ready else \'Model unavailable\' }}</span></div><div id="dropzone" class="dropzone" tabindex="0" role="button" aria-label="Choose or drop a skin photo"><img id="preview" alt="Selected skin photo" hidden><div id="placeholder"><div class="upload-icon"><svg viewBox="0 0 24 24" width="28" height="28" fill="none" stroke="currentColor" stroke-width="1.8" aria-hidden="true"><path d="M12 16V3m-5 5 5-5 5 5M4 15v5a1 1 0 0 0 1 1h14a1 1 0 0 0 1-1v-5"/></svg></div><h3>Drop your image here</h3><p>or click to browse your device</p><span class="file-hint">JPG, PNG or WEBP · Up to 10 MB</span></div></div><input id="fileInput" type="file" accept="image/jpeg,image/png,image/webp" capture="environment" hidden><div class="photo-tips"><div><span>☀</span><b>Natural light</b><small>Avoid harsh shadows</small></div><div><span>⊙</span><b>Keep it centered</b><small>Show the affected area</small></div><div><span>◇</span><b>Sharp and clear</b><small>No filters or glare</small></div></div><p id="fileName" class="small muted"></p><div class="upload-actions"><button id="chooseBtn" class="btn btn-outline" type="button">Choose / take photo</button><button id="analyzeBtn" class="btn" type="button" disabled>Analyze image <span>→</span></button></div><p id="error" class="error" role="alert" hidden></p><p class="privacy-note">Your photo is processed in memory. It is not stored with the report.</p></div><div class="panel result-panel" aria-live="polite">{% include \'result_panel.html\' %}</div></section>\n<div id="loadingOverlay" class="loading-overlay" hidden role="status" aria-live="polite"><div class="loading-card"><div class="loading-orbit">S+</div><p class="eyebrow">SKINSIGHT / IMAGE ANALYSIS</p><h2>A closer look at your photo.</h2><p>Checking the image and preparing your screening result.</p><div class="loading-track"><span></span></div><small>This may take a moment. No diagnosis or stage is being assigned.</small></div></div>\n{% endblock %}{% block scripts %}<script src="/static/app.js"></script>{% endblock %}\n', 'app/templates/home.html': '{% extends \'base.html\' %}{% block title %}SkinSight AI · A clearer next step{% endblock %}\n{% block content %}\n<section class="hero container"><div class="hero-copy"><p class="eyebrow"><span class="tiny-dot"></span> SKIN INSIGHTS, THOUGHTFULLY DESIGNED</p><h1>A clearer picture.<br/><span class="gradient-text">A better next step.</span></h1><p class="lead">Understand what a skin-photo model sees, explore your results, and take useful questions to your dermatologist.</p><div class="actions"><a class="btn" href="/analyze">Analyze a skin photo <span>→</span></a><a class="btn btn-outline" href="#how-it-works">See how it works</a></div><div class="hero-tags"><span>✧ Six lesion categories</span><span>▤ Downloadable reports</span><span>◎ Doctor guidance</span></div></div><div class="hero-visual" aria-label="Illustration of skin-image analysis, not a patient result"><div class="visual-grid"></div><div class="scan-orbit orbit-one"></div><div class="scan-orbit orbit-two"></div><div class="scan-core"><div class="cell c1"></div><div class="cell c2"></div><div class="cell c3"></div><div class="cell c4"></div><div class="cell c5"></div><div class="scan-line"></div></div><div class="float-card top-float"><span class="float-icon">◎</span><div><strong>A closer look</strong><small>Patterns, textures, model scores</small></div></div><div class="float-card bottom-float"><span class="float-icon green">▤</span><div><strong>Keep the conversation going</strong><small>Your report. Your next appointment.</small></div></div><span class="visual-caption">SKINSIGHT / IMAGE INTELLIGENCE</span></div></section>\n<section class="trust-strip container"><div><strong>6</strong><span>Lesion categories</span></div><div><strong>27,137</strong><span>Training & evaluation images</span></div><div><strong>MobileNetV3</strong><span>Deep-learning architecture</span></div><div><strong>For learning</strong><span>Not a diagnostic device</span></div></section>\n<section id="how-it-works" class="section container"><div class="section-heading"><p class="eyebrow">FROM PHOTO TO PERSPECTIVE</p><h2>Simple to use.<br/>Clear about what it means.</h2><p>Three steps to explore a screening result.</p></div><div class="three-grid"><article class="feature-card"><span class="step-number">01</span><h3>Start with a clear photo</h3><p>Use natural light, keep the lesion in focus, and avoid filters. Obvious unusable images are rejected.</p></article><article class="feature-card"><span class="step-number">02</span><h3>Explore the model’s result</h3><p>See the top three categories and their model scores, along with relevant doctor guidance.</p></article><article class="feature-card"><span class="step-number">03</span><h3>Keep your report</h3><p>Download a PDF, find it in your dashboard, or receive it at your registered email when Gmail is configured.</p></article></div></section>\n<section class="container callout"><div><p class="eyebrow">KNOW THE LIMITS</p><h2>Insight starts here.<br/>Diagnosis belongs with a clinician.</h2><p>Our model cannot determine cancer stage from a photo. Learn what staging involves and when to seek professional assessment.</p></div><a class="btn btn-outline" href="/stages">Explore the stage guide <span>→</span></a></section>\n{% endblock %}\n', 'app/templates/stages.html': '{% extends \'base.html\' %}{% block title %}Cancer stage guide · SkinSight AI{% endblock %}{% block content %}<section class="page-heading container"><p class="eyebrow">EDUCATION, NOT A STAGE PREDICTION</p><h1>Understand staging.<br/><span class="gradient-text">Leave diagnosis to your doctor.</span></h1><p class="lead">A skin photograph cannot tell how deep a tumor is or whether it has spread. Clinical staging needs a confirmed diagnosis and additional findings.</p></section><section class="container panel notice"><span class="section-icon">i</span><div><h3>No stage is assigned by this app</h3><p>Model scores, lesion size in a photo and visible color cannot establish stage. This guide is general education, not an assessment of your image.</p></div></section><section class="container section"><div class="section-heading"><p class="eyebrow">MELANOMA / A GENERAL OVERVIEW</p><h2>How melanoma stages are described</h2><p>Within each stage, more detailed subgroups depend on pathology and other clinical findings.</p></div><div class="stage-timeline"><article><span>0</span><div><h3>In situ</h3><p>Melanoma is limited to the outer skin layer.</p></div></article><article><span>I–II</span><div><h3>Localized melanoma</h3><p>Pathology, including tumor thickness and ulceration, helps distinguish localized stages.</p></div></article><article><span>III</span><div><h3>Regional spread</h3><p>Regional lymph nodes or nearby skin and lymphatic sites may be involved.</p></div></article><article><span>IV</span><div><h3>Distant spread</h3><p>Melanoma has spread to distant sites or organs.</p></div></article></div></section><section class="container two-grid"><article class="panel"><h2>Squamous cell carcinoma</h2><p>SCC staging depends on the anatomic site, tumor characteristics and evidence of lymph-node or distant spread. A dermatologist determines the appropriate staging system.</p></article><article class="panel"><h2>Basal cell carcinoma</h2><p>Most BCC cases are assessed by local extent and recurrence risk. Routine numerical staging is uncommon; advanced cases require specialist assessment.</p></article></section><section class="container section panel"><h2>What to discuss at your appointment</h2><p>Ask whether a biopsy is needed, what the pathology shows, whether thickness or ulceration matters, and whether lymph-node assessment or other tests are indicated.</p><a class="text-link" href="https://www.cancer.gov/types/skin/patient/melanoma-treatment-pdq" target="_blank" rel="noopener noreferrer">Read the National Cancer Institute’s melanoma information →</a></section>{% endblock %}\n', 'app/templates/faq.html': '{% extends \'base.html\' %}{% block title %}FAQ · SkinSight AI{% endblock %}{% block content %}<section class="page-heading container narrow"><p class="eyebrow">A LITTLE CLARITY</p><h1>Good questions.<br/><span class="gradient-text">Honest answers.</span></h1><p class="lead">What to expect before you upload—and after you see a result.</p></section><section class="container narrow faq-list">\n{% for question,answer in faqs %}<details class="faq-item"><summary>{{ question }}<span>+</span></summary><p>{{ answer }}</p></details>{% endfor %}\n</section><section class="container narrow callout"><div><h2>Still unsure about a lesion?</h2><p>A new, changing, bleeding, painful or unusual lesion deserves clinical review. Seek advice from a qualified dermatologist.</p></div><a href="/analyze" class="btn">Start a screening <span>→</span></a></section>{% endblock %}\n', 'app/templates/login.html': '{% extends \'base.html\' %}{% block title %}{{ \'Create account\' if register else \'Log in\' }} · SkinSight AI{% endblock %}{% block content %}<section class="auth-layout container"><div class="auth-story"><p class="eyebrow">YOUR SKINSIGHT SPACE</p><h1>A little insight.<br/><span>A thoughtful next step.</span></h1><p>Explore skin-photo screening with results you can understand and reports you can keep.</p><div class="auth-art"><div class="auth-ring"></div><span>S+</span></div><div class="auth-note"><b>Your account. Your reports.</b><p>Screening is a starting point.<br/>Clinical assessment comes next.</p></div></div><div class="auth-form"><p class="eyebrow">{{ \'START YOUR JOURNEY\' if register else \'GOOD TO SEE YOU AGAIN\' }}</p><h2>{{ \'Create an account\' if register else \'Welcome back.\' }}</h2><p class="muted">{{ \'Register to explore skin-photo screening.\' if register else \'Log in to your screening workspace.\' }}</p>{% if error %}<p class="error" role="alert">{{ error }}</p>{% endif %}<form method="post" action="{{ \'/register\' if register else \'/login\' }}"><input type="hidden" name="csrf" value="{{ csrf }}"><label for="username">{{ \'Username\' if register else \'Username or email\' }}</label><input id="username" name="username" required minlength="3" maxlength="{{ 32 if register else 254 }}" {% if register %}pattern="[a-zA-Z0-9_]+"{% endif %} autocomplete="username" class="auth-input" placeholder="{{ \'Choose a username\' if register else \'Your username or email\' }}">{% if register %}<label for="email">Report email</label><input id="email" name="email" type="email" required maxlength="254" autocomplete="email" class="auth-input" placeholder="you@example.com">{% endif %}<label for="password">Password</label><input id="password" name="password" type="password" required minlength="{{ 12 if register else 1 }}" maxlength="128" autocomplete="{{ \'new-password\' if register else \'current-password\' }}" class="auth-input" placeholder="{{ \'At least 12 characters\' if register else \'Your password\' }}">{% if register %}<p class="small muted">Successful analyses send a PDF report and doctor guidance directly to your registered email when Gmail is configured.</p>{% endif %}<button class="btn full-width" type="submit">{{ \'Create account\' if register else \'Log in\' }} <span>→</span></button></form><p class="auth-switch">{{ \'Already have an account?\' if register else \'New to SkinSight?\' }} <a href="{{ \'/login\' if register else \'/register\' }}">{{ \'Log in\' if register else \'Create an account\' }}</a></p><p class="small muted">Educational screening, not a diagnosis or a cancer-stage prediction.</p></div></section>{% endblock %}\n', 'app/templates/about.html': '{% extends \'base.html\' %}{% block title %}About · SkinSight AI{% endblock %}{% block content %}<section class="page-heading container"><p class="eyebrow">THE PROJECT, EXPLAINED</p><h1>Deep learning.<br/><span class="gradient-text">Clear expectations.</span></h1><p class="lead">SkinSight is an educational project combining skin-image classification with a practical screening and reporting workflow.</p></section><section class="container two-grid section"><article class="panel"><span class="section-icon">◎</span><h2>The model behind the screen</h2><p>MobileNetV3-Small is a convolutional neural network. Starting with ImageNet-pretrained features, we trained its classifier, fine-tuned all layers, then adapted it to smartphone skin photos.</p><div class="model-path"><span>ImageNet</span><b>→</b><span>PAD + ISIC</span><b>→</b><span>PAD adaptation</span></div><p class="muted">Six classes: ACK, BCC, MEL, NEV, SCC and SEK. The model does not assess general skin diseases.</p></article><article class="panel"><span class="section-icon">▦</span><h2>Two datasets. Different perspectives.</h2><div class="dataset-row"><div><strong>PAD-UFES-20</strong><p>Clinical smartphone photographs</p></div><strong>2,298</strong></div><div class="dataset-row"><div><strong>ISIC 2019</strong><p>Selected dermoscopy images</p></div><strong>24,839</strong></div><p class="muted">Related recorded patients, lesions and exact-file duplicates stay together when splitting data. ISIC lesion grouping does not guarantee patient independence.</p></article></section><section class="container panel"><p class="eyebrow">RESULTS IN CONTEXT</p><h2>56.9% validation balanced accuracy.</h2><p>The selected model improved from 50.4% on the same smartphone validation cohort. This metric averages recall across six classes. Validation was used to choose the checkpoint, so this is not independent test accuracy.</p><p class="muted">A high score for an individual image is not a calibrated probability of disease. Independent clinical evaluation and calibration are still needed.</p></section><section class="container section three-grid"><article class="feature-card"><h3>Your account, your reports</h3><p>Report downloads are restricted to the account that created them. Uploaded photos are processed in memory; report summaries remain in the host computer’s local database.</p></article><article class="feature-card"><h3>Designed to inform</h3><p>Screening results include doctor guidance and a PDF you can discuss at an appointment. They do not replace examination or a biopsy.</p></article><article class="feature-card"><h3>Quality checks, with limits</h3><p>Obvious graphics and unusable images are rejected. These checks do not prove a skin lesion is present; unrelated photographs can still pass.</p></article></section>{% endblock %}\n', 'app/templates/account.html': '{% extends \'base.html\' %}{% block title %}Email settings · SkinSight AI{% endblock %}{% block content %}<section class="container page-heading narrow"><p class="eyebrow">YOUR ACCOUNT</p><h1>Your report.<br/><span class="gradient-text">Your inbox.</span></h1><p class="lead">Successful analyses send a PDF and doctor guidance directly to your registered email. No uploaded photo is attached.</p></section><section class="container panel account-panel"><h2>Email settings</h2>{% if message %}<p role="status">{{ message }}</p>{% endif %}{% if not mail_ready %}<p class="error">The Gmail sender is not configured. PDF downloads are still available.</p>{% endif %}<form action="/account/email" method="post"><input type="hidden" name="csrf" value="{{ csrf }}"><label for="email">Report email address</label><input id="email" name="email" class="auth-input" type="email" required maxlength="254" value="{{ profile.email or \'\' }}" placeholder="you@example.com"><button class="btn" type="submit">Save email <span>→</span></button></form><p class="small muted">Reports go directly to this address. There is no verification-code step.</p><a class="text-link" href="/analyze">Back to analysis →</a></section>{% endblock %}\n', 'app/templates/base.html': '<!doctype html>\n<html lang="en" data-theme="dark">\n<head><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1"><meta name="csrf-token" content="{{ csrf }}"><title>{% block title %}SkinSight AI{% endblock %}</title><link rel="stylesheet" href="/static/styles.css"><script defer src="/static/site.js"></script></head>\n<body>\n<header class="site-header"><div class="nav-wrap"><a class="brand" href="/home" aria-label="SkinSight home"><span class="brand-mark">S<span>+</span></span>SkinSight<span class="brand-ai">AI</span></a><button id="navToggle" class="icon-button mobile-nav" type="button" aria-label="Toggle navigation" aria-expanded="false">☰</button><nav id="mainNav" aria-label="Main navigation"><a href="/home" {% if page==\'home\' %}aria-current="page"{% endif %}>Home</a><a href="/about" {% if page==\'about\' %}aria-current="page"{% endif %}>About</a><a href="/faq" {% if page==\'faq\' %}aria-current="page"{% endif %}>FAQ</a><a href="/stages" {% if page==\'stages\' %}aria-current="page"{% endif %}>Stage guide</a>{% if username %}<a href="/analyze" {% if page==\'analyze\' %}aria-current="page"{% endif %}>Analyze</a><a href="/dashboard" {% if page==\'dashboard\' %}aria-current="page"{% endif %}>Dashboard</a>{% endif %}</nav><div class="nav-actions"><button id="themeToggle" type="button" class="icon-button" aria-label="Switch color theme">◐</button>{% if username %}<a class="avatar" href="/account" title="Email settings for {{ username }}" aria-label="Email settings for {{ username }}">{{ username[0]|upper }}</a><form action="/logout" method="post"><input type="hidden" name="csrf" value="{{ csrf }}"><button class="btn btn-small btn-outline" type="submit">Log out</button></form>{% else %}<a class="login-link" href="/login">Log in</a><a class="btn btn-small" href="/register">Get started <span>→</span></a>{% endif %}</div></div></header>\n<main>{% block content %}{% endblock %}</main>\n<footer class="site-footer"><div class="footer-grid"><div><a class="brand" href="/home"><span class="brand-mark">S<span>+</span></span>SkinSight<span class="brand-ai">AI</span></a><p>Make your next step an informed one.<br/>Educational skin-photo screening with clear limits.</p></div><div><h4>Explore</h4><a href="/analyze">Analyze a photo</a><a href="/dashboard">Your reports</a><a href="/faq">Questions & answers</a></div><div><h4>Understand</h4><a href="/about">About the project</a><a href="/stages">Cancer stage guide</a><a href="/account">Email settings</a></div><div><h4>A note on your health</h4><p>A model result is not a diagnosis. A qualified dermatologist should assess concerning lesions.</p></div></div><div class="footer-bottom"><span>© {{ year }} SkinSight AI · Educational prototype</span><span>Designed for clarity. Built with care.</span></div></footer>\n{% block scripts %}{% endblock %}</body></html>\n', 'app/templates/result_panel.html': '<div id="emptyResult" class="empty-state"><div class="empty-symbol">◎</div><h3>Your result, clearly explained.</h3><p>Choose a clear skin photo to begin.<br/>Your model scores and next-step guidance will appear here.</p></div>\n<div id="result" hidden><div class="result-heading"><div><p class="eyebrow">MODEL’S TOP CATEGORY</p><h2 id="diagnosis">—</h2><span id="risk" class="result-tag"></span></div><div class="score"><strong id="confidence">0%</strong><span>model score</span></div></div><p id="message" class="result-description"></p><div class="guidance-card"><span class="section-icon">✧</span><div><h3>Recommended next step</h3><p id="doctorText"></p></div></div><h3 class="subheading">Top three predictions</h3><div id="probabilities" class="probabilities"></div><div class="report-actions"><a id="downloadReport" class="btn" href="#">Download PDF report <span>↓</span></a><a href="/dashboard" class="btn btn-outline">View your dashboard</a></div><p id="emailStatus" class="small muted" aria-live="polite"></p><details class="stage-result"><summary>Stage information <span>+</span></summary><p><strong id="stageStatus"></strong></p><p id="stageText"></p><div id="stageGuide"></div><a href="/stages" class="text-link">Read the stage guide →</a></details><p class="disclaimer" id="disclaimer"></p></div>\n', 'app/templates/report.html': '{% extends \'base.html\' %}{% block title %}Your screening report · SkinSight AI{% endblock %}{% block content %}<section class="container page-heading"><a class="text-link" href="/dashboard">← Back to dashboard</a><p class="eyebrow">SAVED SCREENING RESULT</p><h1>A result you can<br/><span class="gradient-text">take with you.</span></h1><p class="muted">Created {{ report.created_at }} · No original image is stored.</p></section><section class="container report-view panel">{% include \'result_panel.html\' %}</section>{% endblock %}{% block scripts %}<script>window.savedReport = {{ report|tojson }};</script><script src="/static/app.js"></script>{% endblock %}\n', 'app/templates/dashboard.html': '{% extends \'base.html\' %}{% block title %}Your dashboard · SkinSight AI{% endblock %}{% block content %}<section class="dashboard-heading container"><div><p class="eyebrow">YOUR SCREENING ACTIVITY</p><h1>Good to see you,<br/><span class="gradient-text">{{ username }}.</span></h1><p class="lead">Your saved results and reports, in one place.</p></div><a class="btn" href="/analyze">New analysis <span>＋</span></a></section>\n<section class="container stats-grid"><article class="stat-card"><span class="stat-icon">▤</span><small>Total analyses</small><strong>{{ stats.total }}</strong><p>Saved reports</p></article><article class="stat-card"><span class="stat-icon cyan">◷</span><small>Today (UTC)</small><strong>{{ stats.today }}</strong><p>Recent activity</p></article><article class="stat-card"><span class="stat-icon green">◎</span><small>Average model score</small><strong>{{ (\'%.1f\'|format(stats.average)) ~ \'%\' if stats.total else \'—\' }}</strong><p>Not diagnostic certainty</p></article><article class="stat-card"><span class="stat-icon amber">!</span><small>Cancer-class predictions</small><strong>{{ stats.cancer_classes }}</strong><p>MEL, BCC or SCC · not diagnoses</p></article></section>\n{% if stats.total %}<section class="container two-grid dashboard-charts"><article class="panel"><h3>Activity over time</h3><p class="muted small">Reports from the last six calendar months (UTC)</p><div class="bar-chart" role="img" aria-label="Monthly counts of your reports">{% for month in months %}<div class="bar-column"><b>{{ month.count }}</b><div class="bar-track"><span style="height:{{ month.height }}%"></span></div><small>{{ month.label }}</small></div>{% endfor %}</div></article><article class="panel"><h3>Category mix</h3><p class="muted small">Model predictions, not confirmed conditions</p><div class="frequency-list">{% for item in frequencies %}<div><span class="category-label">{{ item.label }}</span><div class="track"><span style="width:{{ item.percent }}%"></span></div><b>{{ item.count }}</b></div>{% endfor %}</div></article></section>{% endif %}\n<section class="container panel activity-panel"><div class="panel-title"><div><h2>Your reports</h2><p class="muted">Results from your account only. Images are not stored.</p></div><span class="status-pill">{{ stats.total }} saved</span></div>{% if reports %}<div class="table-scroll"><table class="activity-table"><thead><tr><th>Observation</th><th>Model score</th><th>Created (UTC)</th><th>Email</th><th>Report</th></tr></thead><tbody>{% for report in reports %}<tr><td><a href="/reports/{{ report.report_id }}">{{ report.guidance.name }}</a><small>{{ report.label }} · MobileNetV3-Small</small></td><td>{{ \'%.1f\'|format(report.confidence) }}%</td><td>{{ report.display_date }}</td><td><span class="status-text">{{ report.email_description }}</span></td><td><a class="text-link" href="/reports/{{ report.report_id }}">View →</a><a class="download-link" href="/api/reports/{{ report.report_id }}/download">PDF ↓</a></td></tr>{% endfor %}</tbody></table></div>{% else %}<div class="empty-state compact"><span class="empty-symbol">▤</span><h3>Your story starts with one photo.</h3><p>No saved analyses yet. Results will appear here after screening.</p><a class="btn" href="/analyze">Start your first analysis →</a></div>{% endif %}</section><div class="container dashboard-note"><p>Model scores are uncalibrated. A dashboard trend cannot establish changes in a lesion or cancer stage.</p><a class="text-link" href="/about">Understand the model →</a></div>{% endblock %}\n', 'reports/finetune_v1/RESULTS.md': '# Full-model fine-tuning results\n\nSmartphone (PAD) validation balanced accuracy improved from 50.4% to 56.9%. This is validation performance used to choose the checkpoint, not independent test accuracy.\n\n- mobilenet_finetune_v1: 52.9% PAD validation balanced accuracy, best epoch 5, fitting source combined, 16330 training images.\n- mobilenet_pad_adapt_v1: 56.9% PAD validation balanced accuracy, best epoch 3, fitting source PAD-UFES-20, 1380 training images.\n\nSelected model: mobilenet_pad_adapt_v1. SHA256: `a4619d1b9f73cdf2fa415e285642878472356cdd3a135dd8e6cd316d629db0d1`.\n\nCombined warm-start fine-tuning: 5 epochs; PAD-only adaptation: 8 epochs. Both use all trainable layers, learning rate 0.00003, batch size 32, image size 160, seed 42, CPU, augmentation, and the same fixed original partitions. Checkpoint selection uses PAD validation. Original test images were not evaluated in either new run.\n\nHistorical baseline test results (49.6% combined; 52.6% PAD) apply only to the old frozen-feature model. They do not measure this model. External clinical evaluation and actual temperature calibration remain pending a new eligible dataset. Calibration tooling is implemented but no real-data temperature has been fitted. ISIC lesion grouping does not prove patient independence. This model remains an educational prototype.\n\nDDI access requires individual registration and research-use terms at https://ddi-dataset.github.io/#access. It has not been downloaded or tested. Confirm label mapping and patient identifiers before defining separate calibration and independent evaluation cohorts. Do not substitute unsupported labels.\n', 'reports/baseline_v1/RESULTS.md': '# Real-data baseline results\n\nA five-epoch frozen-feature MobileNetV3-Small baseline was trained on the complete\nsix-class dataset (27,137 matched images), with no train-time augmentation.\nThe backbone remained at pretrained ImageNet weights; only the classifier was fitted.\nEpoch 4 was selected by validation balanced accuracy.\nThe fixed test partition was evaluated after checkpoint selection, without tuning\nagainst test results. This is a research prototype, not a reliable medical classifier.\n\n| Held-out source | Images | Balanced accuracy |\n|---|---:|---:|\n| Combined | 5335 | 49.6% |\n| PAD smartphone photographs | 436 | 52.6% |\n| ISIC dermoscopy | 4899 | 47.4% |\n\nBalanced accuracy is the mean of class sensitivities, not ordinary accuracy.\nThe PAD melanoma test contains only 5 images: 3/5 were correctly classified.\nThat sample is far too small for dependable melanoma performance claims.\nScores are uncalibrated; external smartphone accuracy remains unmeasured.\nOfficial ISIC metadata enables lesion grouping where present, not guaranteed patient\nindependence. Linked identities and exact-file hashes were verified disjoint across\ntrain/validation/test. Re-encoded or cropped duplicates are not ruled out.\n\nThe actual checkpoint loaded successfully in the app. A held-out PAD image returned\nHTTP 200 with the expected prediction, guidance, and staging limitations. The home\npage and health endpoint passed. Thirteen software regression tests passed.\nThe checkpoint is intentionally excluded from Git and supplied in the downloadable\ntrained-app ZIP and prepared environment filesystem.\n\nTraining source commit: `63a917f782d16354265c4ce1205d0f8630fd2495`.\nCheckpoint SHA-256: `e8d79d8600324d90e96a37f55163c653060b23691608a907e4c12dc5b1c6b5fb`.\nSplit summary and complete aggregate validation/test reports accompany this file.\nNo individual patient records or test image paths are included in Git reports.\n\nNext research steps: fine-tune using validation only, obtain genuine ISIC patient\nmapping where possible, establish a new independent external test, and assess\ncalibration. Do not tune further using this already-reported test partition.\n'}
for name, contents in FILES.items():
    destination = PROJECT / name
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(contents, encoding='utf-8')
print('Prepared corrected project:', PROJECT)


In [ ]:
import kagglehub
pad = Path(kagglehub.dataset_download('mahdavi1202/skin-cancer'))
isic = Path(kagglehub.dataset_download('nischaydnk/isic-2019-jpg-224x224-resized'))
gt_download = Path(kagglehub.dataset_download('andrewmvd/isic-2019', path='ISIC_2019_Training_GroundTruth.csv'))
import urllib.request
isic_meta = PROJECT / 'data/ISIC_2019_Training_Metadata.csv'
isic_meta.parent.mkdir(parents=True, exist_ok=True)
with urllib.request.urlopen('https://isic-challenge-data.s3.amazonaws.com/2019/ISIC_2019_Training_Metadata.csv', timeout=120) as response:
    isic_meta.write_bytes(response.read())
def locate(root, name):
    if root.is_file() and root.name == name:
        return root
    matches = list(root.rglob(name))
    if len(matches) != 1:
        raise RuntimeError(f'Expected exactly one {name} in {root}; found {len(matches)}')
    return matches[0]
pad_meta = locate(pad, 'metadata.csv')
isic_gt = locate(gt_download, 'ISIC_2019_Training_GroundTruth.csv')
print('PAD:',pad, 'ISIC:',isic, 'ground truth:',isic_gt)


In [ ]:
import pandas as pd
manifest = PROJECT / 'data/combined_manifest.csv'
subprocess.run([sys.executable, 'ml/prepare_combined.py',
    '--pad-meta', str(pad_meta), '--pad-root', str(pad),
    '--isic-gt', str(isic_gt), '--isic-root', str(isic),
    '--isic-meta', str(isic_meta),
    '--out', str(manifest)], cwd=PROJECT, check=True)
frame = pd.read_csv(manifest)
counts = frame.groupby('source').size().to_dict()
assert counts == {'PAD-UFES-20': 2298, 'ISIC-2019': 24839}, counts
print('Verified all 27,137 supported records matched images.')


In [ ]:
# Baseline training; ten epochs do not guarantee a clinically useful model.
subprocess.run([sys.executable, 'ml/train.py', '--manifest', str(manifest),
    '--epochs', '10', '--batch-size', '32', '--workers', '2', '--size', '160',
    '--fine-tune', '--lr', '0.00003', '--selection-source', 'PAD-UFES-20', '--skip-test', '--device', 'cuda', '--output-dir', str(PROJECT / 'models')],
    cwd=PROJECT, check=True)


In [ ]:
import json, shutil
print('Validation:', (PROJECT/'models/metrics.json').read_text())
print('Old reported test was deliberately not evaluated.')
print('Grouping coverage:', (PROJECT/'models/split_summary.json').read_text())
# Check loading and prediction mechanics against a held-out image.
# This is software validation, not an independent accuracy estimate.
sys.path.insert(0,str(PROJECT))
os.environ['SKINSIGHT_MODEL_PATH'] = str(PROJECT/'models/skinsight_mobilenetv3_small.pt')
from app.inference import Predictor
predictor = Predictor()
assert predictor.ready
held_out = pd.read_csv(PROJECT/'models/validation_split.csv')
print('Functional prediction:', predictor.predict_bytes(Path(held_out.iloc[0].image_path).read_bytes()))
# Include app and trained checkpoint, omit image paths/manifests referring to Kaggle.
export = Path('/kaggle/working/skinsight_export')
if export.exists():
    raise RuntimeError('Export directory exists; choose a new path to preserve previous results.')
shutil.copytree(PROJECT, export/'SkinSight_AI', ignore=shutil.ignore_patterns('data','__pycache__','*_split.csv'))
archive = shutil.make_archive('/kaggle/working/skinsight_trained_app', 'zip', export)
print('Download this file from notebook output:', archive)


Download `skinsight_trained_app.zip` from the notebook output. Unzip on Windows, open PowerShell in its `SkinSight_AI` folder, and follow README installation/run steps. Metrics include source-specific validation. The old reported test is not evaluated again. Patient independence depends on grouping coverage. Do not tune using the held-out test or use this model for clinical decisions.